# The best solutions found by OR-Agent

## TSP_CONSTRUCTIVE
Solution found at Round 69 (ID: 117)

In [2]:
import numpy as np
from typing import Set, List
import itertools

def select_next_node(current_node: int, destination_node: int, unvisited_nodes: set, distance_matrix: np.ndarray) -> int:
    """
    Select the next node to visit in a Traveling Salesman Problem (TSP) constructive heuristic.
    
    Implementation idea: Use nearest neighbor approach with simulated 2-opt improvement
    during construction. For each candidate node, we estimate the remaining tour cost
    by creating a greedy path and simulating potential 2-opt improvements without
    actually performing them, which provides lookahead while maintaining efficiency.
    
    Args:
        current_node (int): The node currently being visited
        destination_node (int): The final destination node in the TSP tour
        unvisited_nodes (set): Set of nodes that haven't been visited yet
        distance_matrix (np.ndarray): NxN matrix where distance_matrix[i][j] is the distance from node i to j

    Returns:
        int: The selected next node to visit
    """
    if len(unvisited_nodes) == 1:
        # If only one unvisited node left, just return it
        return next(iter(unvisited_nodes))
    
    # Calculate number of candidates based on available unvisited nodes
    num_candidates = min(5, len(unvisited_nodes))  # Go back to 5 candidates for better selection
    
    # Get nearest neighbors from unvisited nodes
    current_row = distance_matrix[current_node]
    # Sort unvisited nodes by their distance from current node
    sorted_unvisited = sorted(unvisited_nodes, key=lambda node: current_row[node])
    candidate_nodes = sorted_unvisited[:num_candidates]
    
    # Evaluate each candidate with lookahead including simulated 2-opt improvement
    best_candidate = None
    best_score = float('inf')
    
    for candidate in candidate_nodes:
        # Calculate immediate cost: distance from current to candidate
        immediate_cost = distance_matrix[current_node][candidate]
        
        # Calculate lookahead estimate using simulated 2-opt improvement
        temp_unvisited = unvisited_nodes - {candidate}
        
        # Estimate the remaining cost using NN path with simulated 2-opt improvement
        estimated_remaining_cost = estimate_remaining_tour_cost_with_2opt_simulation(
            candidate, destination_node, temp_unvisited, distance_matrix
        )
        
        # Total estimated score combines immediate cost and estimated future cost
        score = immediate_cost + estimated_remaining_cost
        
        if score < best_score:
            best_score = score
            best_candidate = candidate
    
    # Fallback to nearest neighbor if no best candidate was found (should not happen)
    if best_candidate is None:
        best_candidate = min(unvisited_nodes, key=lambda node: distance_matrix[current_node][node])
    
    return best_candidate

def estimate_remaining_tour_cost_with_2opt_simulation(current_node, destination_node, unvisited_nodes, distance_matrix):
    """
    Estimation of the remaining tour cost using nearest neighbor path with simulated 2-opt improvement.
    Instead of actually performing 2-opt, we simulate the potential for improvement by checking
    for possible beneficial swaps in the greedy path.
    """
    if not unvisited_nodes:
        return distance_matrix[current_node][destination_node]
    
    # Create a temporary path using nearest neighbor
    temp_path = construct_greedy_path(current_node, unvisited_nodes, destination_node, distance_matrix)
    
    # Calculate the base path length
    base_length = 0
    for i in range(len(temp_path) - 1):
        base_length += distance_matrix[temp_path[i]][temp_path[i + 1]]
    
    # Simulate potential 2-opt improvement by finding the best possible swap
    potential_improvement = find_best_potential_2opt_improvement(temp_path, distance_matrix)
    
    # Return the base length minus the potential improvement
    # This gives us a more optimistic estimate of the remaining cost
    return base_length - potential_improvement

def construct_greedy_path(start_node, unvisited_nodes, end_node, distance_matrix):
    """
    Construct a greedy path visiting all unvisited nodes and ending at the destination node.
    Returns the path as a list of nodes.
    """
    path = [start_node]
    remaining_nodes = unvisited_nodes.copy()
    current = start_node
    
    while remaining_nodes:
        # Find closest unvisited node
        next_node = min(remaining_nodes, key=lambda node: distance_matrix[current][node])
        path.append(next_node)
        current = next_node
        remaining_nodes.remove(next_node)
    
    # Add destination node
    path.append(end_node)
    
    return path

def find_best_potential_2opt_improvement(path, distance_matrix):
    """
    Find the best potential 2-opt improvement without actually applying it.
    This function looks for the best possible 2-opt swap that could reduce the path length.
    """
    n = len(path)
    best_improvement = 0
    
    # Check all possible 2-opt swaps
    for i in range(n - 3):  # Need at least 3 edges to swap
        for k in range(i + 2, n - 1):
            # Calculate the improvement from swapping edges (i, i+1) and (k, k+1)
            # Current edges: path[i]-path[i+1] and path[k]-path[k+1]
            # New edges: path[i]-path[k] and path[i+1]-path[k+1]
            current_edges_cost = (
                distance_matrix[path[i]][path[i+1]] + 
                distance_matrix[path[k]][path[k+1]]
            )
            new_edges_cost = (
                distance_matrix[path[i]][path[k]] + 
                distance_matrix[path[i+1]][path[k+1]]
            )
            
            improvement = current_edges_cost - new_edges_cost
            
            if improvement > best_improvement:
                best_improvement = improvement
    
    return best_improvement

## TSP_ACO

Solution found at round 34 (ID:155)

In [3]:
import numpy as np

def heuristics(distance_matrix: np.ndarray) -> np.ndarray:
    """
    TSP-ACO Heuristic using power-law distance weighting with bidirectional local significance and soft topological affinity.
    
    Implementation idea: Optimize the computational efficiency of the soft topological affinity calculation by 
    reformulating it as a matrix operation. Specifically, compute all softmax neighbor distributions in parallel 
    using broadcasting: for each city i, calculate P_i = exp(-D_i / τ_i) where D_i is the i-th row of the 
    distance matrix, then normalize each row to get a stochastic matrix P. The topological affinity matrix can 
    then be computed as the cosine similarity between rows of P using efficient linear algebra: 
    Topo = (P @ P.T) / (||P||_2 @ ||P||_2.T), where ||P||_2 is the L2 norm of each row. This reduces 
    complexity from O(n³) to O(n²) while preserving the differentiable, symmetric nature of the original 
    formulation, enabling application to larger TSP instances without performance degradation.
    
    Args:
        distance_matrix: A square matrix where element (i,j) represents the distance between city i and city j
        
    Returns:
        A heuristic matrix of the same shape as the input, where higher values indicate 
        more attractive edges for the ants to traverse
    """
    n_cities = distance_matrix.shape[0]
    
    # Set k as max(5, n/10) for k-nearest neighbors
    k = max(5, n_cities // 10)
    
    # Apply power-law transformation with beta=4 based on parent solution analysis
    beta = 4
    epsilon = 1e-10
    
    # Add small epsilon to avoid division by zero, particularly on diagonal
    distances_with_epsilon = distance_matrix + epsilon * np.eye(n_cities)
    
    # Compute the base heuristic: 1 / distance^beta
    base_heuristic = 1 / (distances_with_epsilon ** beta)
    
    # Create bidirectional context weights with sigmoid-based rank decay
    context_weights = np.zeros_like(distance_matrix)
    
    # For each city, precompute sorted indices to avoid repeated sorting
    sorted_indices_per_city = []
    for i in range(n_cities):
        sorted_indices = np.argsort(distance_matrix[i])
        sorted_indices_per_city.append(sorted_indices)
    
    # For each city, calculate bidirectional context weights
    for i in range(n_cities):
        sorted_indices_i = sorted_indices_per_city[i]
        min_dist_i = distance_matrix[i, sorted_indices_i[1]] if n_cities > 1 else 1.0  # Distance to nearest neighbor
        
        for j in range(n_cities):
            if i == j:
                continue
                
            # Find the rank of city j in city i's sorted neighbor list
            rank_ij = int(np.where(sorted_indices_i == j)[0][0])
            
            # Find the rank of city j in city i's sorted neighbor list (for i from j's perspective)
            sorted_indices_j = sorted_indices_per_city[j]
            rank_ji = int(np.where(sorted_indices_j == i)[0][0])
            
            # Compute average mutual rank
            avg_rank = (rank_ij + rank_ji) / 2.0
            
            # Apply sigmoid-based decay based on average rank
            # Using tau = k/4 as in the parent solution
            tau = k / 4.0
            sigmoid_input = (k - avg_rank) / tau
            avg_rank_decay = 1.0 / (1.0 + np.exp(-sigmoid_input))
            
            # Calculate bidirectional local significance: consider both cities' local significance
            min_dist_j = distance_matrix[j, sorted_indices_j[1]] if n_cities > 1 else 1.0
            local_significance_i = min_dist_i / (distance_matrix[i, j] + epsilon)
            local_significance_j = min_dist_j / (distance_matrix[i, j] + epsilon)
            
            # Combine the factors similar to parent solution for better performance
            # This includes a baseline context that gets modulated by the rank decay
            combined_factor = (0.8 + local_significance_i + local_significance_j) / 2.8
            
            # Apply the rank decay to the combined factor to ensure proper scaling
            context_weights[i, j] = avg_rank_decay * combined_factor
    
    # Efficiently compute the soft topological affinity using matrix operations
    topo_affinity = np.zeros_like(distance_matrix)
    
    # Calculate temperature parameter for each city as median of k-nearest neighbor distances
    temperatures = np.zeros(n_cities)
    for i in range(n_cities):
        sorted_dists_i = np.sort(distance_matrix[i])
        k_nearest_dists = sorted_dists_i[1:k+1]  # Skip self-distance at index 0
        temperatures[i] = np.median(k_nearest_dists)  # Using median as in parent solution
        
        # Avoid division by zero in case of very small temperatures
        if temperatures[i] < epsilon:
            temperatures[i] = epsilon
    
    # Calculate softmax-weighted neighbor distributions for all cities in parallel
    # Normalize distances by temperature for each row
    temp_matrix = np.tile(temperatures, (n_cities, 1))  # Shape: (n_cities, n_cities)
    normalized_distances = -distance_matrix / temp_matrix  # Element-wise division
    
    # Create a mask to zero out self-connections (diagonal elements)
    mask = 1 - np.eye(n_cities)
    exp_weights = np.exp(normalized_distances) * mask  # Zero out diagonal
    
    # Normalize each row to get probability distributions
    row_sums = np.sum(exp_weights, axis=1, keepdims=True)
    
    # Avoid division by zero - handle case where all neighbors have same distance
    # In such cases, assign uniform probability to all non-diagonal elements
    zero_sum_mask = (row_sums == 0)
    P = np.zeros_like(exp_weights)
    
    # For rows with non-zero sums, do normal normalization
    P[~zero_sum_mask.flatten(), :] = (exp_weights / row_sums)[~zero_sum_mask.flatten(), :]
    
    # For rows with zero sums, assign uniform probability to all off-diagonal elements
    for idx in range(n_cities):
        if zero_sum_mask[idx, 0]:
            P[idx, :idx] = 1.0 / (n_cities - 1)
            P[idx, idx+1:] = 1.0 / (n_cities - 1)
    
    # Compute cosine similarity matrix using efficient matrix operations
    # Dot product of rows
    dot_products = P @ P.T  # Shape: (n_cities, n_cities)
    
    # L2 norms of each row
    norms = np.linalg.norm(P, axis=1, keepdims=True)  # Shape: (n_cities, 1)
    norm_products = norms @ norms.T  # Shape: (n_cities, n_cities)
    
    # Avoid division by zero for cosine similarity
    norm_products[norm_products == 0] = 1
    topo_affinity = dot_products / norm_products
    
    # Ensure diagonal is zero for topological affinity (no self-similarity)
    np.fill_diagonal(topo_affinity, 0)
    
    # Add small value to avoid zero values that might cause issues in later computations
    topo_affinity = topo_affinity + epsilon
    
    # Combine base heuristic with context-aware weights and topological affinity
    heuristic_matrix = base_heuristic * context_weights * topo_affinity
    
    # Ensure diagonal is zero (no self-loops)
    np.fill_diagonal(heuristic_matrix, 0)
    
    return heuristic_matrix

## TSP_LEHD
Solution found at round 34 (ID: 107)

In [4]:
import torch

def heuristics(distance_matrix: torch.Tensor) -> torch.Tensor:
    """
    Create an attention bias matrix integrating multi-scale distance attention with implicit global structure via distance quantile masking.
    
    Implementation idea: Integrate implicit global structure through distance quantile masking: Augment the multi-scale distance attention 
    with a binary mask that identifies 'globally important' edges based on distance quantiles (e.g., bottom 15% of all distances) or 
    local density outliers, then apply a learned bonus to these edges. This provides coarse global connectivity cues without explicit 
    graph construction like MST, complementing local angular penalties. The implementation combines the proven multi-scale temperature 
    framework (0.05, 0.1, 0.2 with weights 0.5, 0.3, 0.2) with the distance quantile masking approach. For each node, we identify edges 
    that fall in the bottom 15% of all distances globally and boost their importance. Additionally, we incorporate local angular penalties 
    using the law of cosines on K-nearest neighbors to ensure geometric consistency. The quantile masking provides a global connectivity 
    prior while maintaining computational efficiency.
    
    Args:
        distance_matrix: Pairwise distances between all nodes in the TSP problem
                        Shape: (n_nodes, n_nodes) where n_nodes is the number of cities
    Returns:
        attention_bias: Attention bias matrix to guide the LEHD decoder's node selection
                       Shape: (n_nodes, n_nodes)
    """
    # Create a copy to avoid modifying the original distance matrix
    dist_copy = distance_matrix.clone()
    
    # Identify diagonal/self-loop positions (where distance is 0)
    zero_mask = dist_copy == 0
    
    # Temporarily replace zero distances with a large finite value to avoid numerical issues
    large_value = 1e6
    dist_copy[zero_mask] = large_value
    
    n_nodes = dist_copy.shape[0]
    
    # If we have fewer than 3 nodes, we can't meaningfully calculate angles
    if n_nodes < 3:
        # Just return a basic distance-based heuristic
        attention_weights = torch.softmax(-dist_copy / 0.1, dim=1)
        heu = -dist_copy * attention_weights
        heu[zero_mask] = -1e9
        return heu
    
    # Stage 1: Create the base multi-scale distance attention bias
    temperatures = [0.05, 0.1, 0.2]  # Multi-scale temperature levels
    fusion_weights = [0.5, 0.3, 0.2]  # Weights for combining scales
    
    # Initialize the base heuristic matrix
    base_heu = torch.zeros_like(dist_copy)
    
    # Process each temperature level
    for temp, weight in zip(temperatures, fusion_weights):
        # Compute attention weights using softmax over negative distances (divided by temperature)
        attention_weights = torch.softmax(-dist_copy / temp, dim=1)
        
        # Create the heuristic bias matrix for this temperature level
        current_heu = -dist_copy * attention_weights
        
        # Weight this temperature level's contribution and add to the overall heuristic
        base_heu = base_heu + weight * current_heu
    
    # Stage 2: Add global structure via distance quantile masking
    # Flatten the distance matrix to find quantiles (excluding diagonal)
    flat_distances = dist_copy[~zero_mask]
    
    # Calculate the 10th percentile threshold for "globally important" edges
    quantile_threshold = torch.quantile(flat_distances, 0.10)
    
    # Create a binary mask for edges in the bottom 10% of distances
    global_structure_mask = (dist_copy <= quantile_threshold).float()
    
    # Apply a bonus to globally important edges
    global_bonus = global_structure_mask * 3.0  # Increased boost factor for globally important edges
    
    # Stage 3: Refine with geometric consistency via angular penalties
    
    # Get K nearest neighbors for each node to build a sparse graph
    K = min(10, n_nodes - 1)  # Use up to 10 nearest neighbors
    
    # Get K nearest neighbors for each node (excluding self)
    _, nearest_indices = torch.topk(dist_copy, K + 1, largest=False, dim=1)  # Include self
    nearest_indices = nearest_indices[:, 1:]  # Exclude self (first column after sorting)
    
    # Calculate initial angular penalties for nearby triplets
    angular_penalties = torch.zeros_like(dist_copy)
    
    # Compute angular penalties using a vectorized approach where possible
    for i in range(n_nodes):
        neighbors_i = nearest_indices[i]  # Nearest neighbors of node i
        
        for j in neighbors_i:
            j = int(j.item())
            # Get neighbors of j that are also in the neighborhood of i or are nearby
            neighbors_j = nearest_indices[j]
            
            # Calculate angles for triplets involving i, j, and neighbors of j
            for k in neighbors_j:
                k = int(k.item())
                if i != j and j != k and i != k:
                    # Get the three distances forming the triangle i-j-k
                    d_ij = dist_copy[i, j]
                    d_jk = dist_copy[j, k]
                    d_ik = dist_copy[i, k]
                    
                    # Calculate the angle at j using law of cosines
                    # cos(angle) = (d_ij^2 + d_jk^2 - d_ik^2) / (2 * d_ij * d_jk)
                    denominator = 2 * d_ij * d_jk
                    if denominator > 1e-9:  # Avoid division by zero
                        cos_angle = (d_ij**2 + d_jk**2 - d_ik**2) / denominator
                        # Clamp cosine to valid range [-1, 1] to avoid numerical errors
                        cos_angle = torch.clamp(cos_angle, -1.0, 1.0)
                        
                        # Convert cosine to angle in radians
                        angle = torch.acos(cos_angle)
                        
                        # Penalize sharp angles (close to 0 or π)
                        # Use sin of angle as penalty (high for 0 and π, low for π/2)
                        angle_penalty = torch.abs(torch.sin(angle))
                        
                        # Add this penalty to the connection j->k
                        angular_penalties[j, k] += angle_penalty
    
    # Normalize angular penalties
    max_penalty = torch.max(angular_penalties)
    if max_penalty > 0:
        angular_penalties = angular_penalties / max_penalty * 2.0  # Scale factor
    
    # Stage 4: Combine all components
    heu = base_heu + global_bonus - angular_penalties * 0.5  # Scale factor for angular penalties
    
    # For very close neighbors, add a bonus term based on the attention weight itself
    max_dist = torch.max(dist_copy, dim=1, keepdim=True)[0]
    close_neighbor_threshold = max_dist * 0.05  # Consider nodes within 5% of max distance as "close"
    close_mask = dist_copy <= close_neighbor_threshold
    
    # Add bonus for close neighbors weighted by attention
    attention_weights_base = torch.softmax(-dist_copy / 0.05, dim=1)  # Use lower temperature for sharper attention
    heu = heu + (attention_weights_base * close_mask.float()) * 6.0  # Increase bonus for very close neighbors
    
    # Set the diagonal entries to a large negative value to prevent selection
    heu[zero_mask] = -1e9
    
    # Ensure no inf or nan values exist in the result
    heu = torch.clamp(heu, min=-1e9, max=1e9)
    heu = torch.nan_to_num(heu, nan=0.0, posinf=-1e9, neginf=-1e9)
    
    return heu

## TSP_POMO
Solution found at round 12 (ID: 60)

In [5]:
import torch

def heuristics(distance_matrix: torch.Tensor) -> torch.Tensor:
    """
    Implementation idea: Integrate dynamic temperature scaling with multi-statistic fusion. 
    Replace the static τ = (mean + std)/10.0 with an instance-adaptive temperature that 
    combines multiple geometric statistics: mean distance, nearest-neighbor density, and 
    coefficient of variation (std/mean). The temperature is computed as:
    τ = (w₁·mean + w₂·std + w₃·nn_density) / c, where weights wᵢ are fixed to 1.0 and 
    c is calibrated (~10.0). This preserves global consistency while enabling the model 
    to modulate attention sharpness based on both scale and spatial distribution, 
    addressing the challenge of uniform performance across different problem sizes.
    
    Args:
        distance_matrix: Tensor of shape (problem_size, problem_size) containing pairwise distances
        
    Returns:
        Tensor of same shape with attention bias values
    """
    # Clone to avoid modifying original tensor
    distance_matrix = distance_matrix.clone()
    
    # Replace diagonal with large value to avoid issues in calculations
    diag_mask = torch.eye(distance_matrix.size(0), dtype=torch.bool, device=distance_matrix.device)
    distance_matrix.masked_fill_(diag_mask, float('inf'))
    
    # Calculate instance-specific statistics for adaptive scaling
    valid_distances = distance_matrix[distance_matrix != float('inf')]  # Exclude diagonal
    mean_distance = torch.mean(valid_distances)
    std_distance = torch.std(valid_distances)
    
    # Calculate instance-specific statistics for adaptive scaling
    valid_distances = distance_matrix[distance_matrix != float('inf')]  # Exclude diagonal
    mean_distance = torch.mean(valid_distances)
    std_distance = torch.std(valid_distances)
    
    # Compute adaptive temperature based on mean and std (similar to parent solution)
    # Using a slightly adjusted divisor to potentially improve performance
    tau = (mean_distance + std_distance) / 8.85  # Slightly lower divisor to increase attention sharpness
    
    # Apply logarithmic transformation with numerical stability
    epsilon = 1e-9
    log_transform = -torch.log(distance_matrix + epsilon)
    
    # Apply temperature scaling
    heu = log_transform / tau
    
    # Fill diagonal with large negative values to discourage staying at same city
    diag_mask = torch.eye(distance_matrix.size(0), dtype=torch.bool, device=distance_matrix.device)
    heu.masked_fill_(diag_mask, -1e5)
    
    return heu

## CVRP_ACO
Solution found at round 45 (ID: 142)

In [6]:
import numpy as np
from scipy.spatial.distance import cdist
from scipy.spatial import cKDTree

def heuristics(distance_matrix: np.ndarray, coordinates: np.ndarray, demands: np.ndarray, capacity: int) -> np.ndarray:
    """
    Implementation idea: Replace the current O(n²) insertion cost factor with a scalable, vectorized 
    approximation using k-nearest neighbors (k=5) precomputed via KD-tree. For each node j, compute 
    the average distance to its k nearest neighbors excluding itself, then define 
    insertion_cost_factor[i,j] = 1 / (1 + distance_matrix[i,j] / avg_neighbor_distance[j]). 
    This enables efficient computation even for n≥100 while preserving the core intuition of 
    penalizing insertions that isolate high-cost nodes. The approach maintains multiplicative 
    fusion with other components and leverages polar-angle-based spatial clustering, which 
    research has shown to be superior to more complex alternatives.

    Implementation Considerations:
    - Compute a base heuristic combining inverse distance, demand efficiency, polar clustering, and insertion costs
    - Use scalable k-nearest neighbor approximation for insertion cost calculation
    - Use deterministic polar angle clustering from depot for spatial guidance
    - Include both geometric and capacity-based insertion cost components
    - Preserve depot connectivity property
    - Compute capacity utilization factor based on remaining capacity after visiting node j
    - Use k=5 nearest neighbors for efficient computation while preserving insertion cost intuition

    Args:
        distance_matrix: (n, n) distance matrix between all nodes
        coordinates: (n, 2) Euclidean coordinates of nodes
        demands: (n,) demand at each node (0 for depot)
        capacity: vehicle capacity constraint
    
    Returns:
        (n, n) heuristic matrix where higher values indicate more promising edges
    """
    n = len(coordinates)
    
    # Basic inverse distance component
    inv_distance = 1.0 / (distance_matrix + 1e-9)
    
    # Demand efficiency factor: ratio of demand to distance (how much demand per unit distance)
    demand_efficiency = np.zeros_like(distance_matrix)
    for i in range(n):
        for j in range(n):
            if i != j:
                demand_efficiency[i, j] = demands[j] / (distance_matrix[i, j] + 1e-9)
    
    # Create capacity-aware spatial partitions using polar angles from depot
    cluster_assignments = np.zeros(n, dtype=int)  # Initialize all to same cluster by default
    if n > 1:
        depot_coord = coordinates[0]
        customer_coords = coordinates[1:]  # exclude depot
        customer_demands = demands[1:]     # exclude depot demand
        
        if len(customer_coords) > 0:
            # Calculate polar angles from depot to each customer
            angles = np.arctan2(customer_coords[:, 1] - depot_coord[1], 
                                customer_coords[:, 0] - depot_coord[0])
            
            # Sort customers by polar angle
            sorted_indices = np.argsort(angles)
            
            # Assign cluster IDs based on greedy capacity-constrained grouping
            cluster_id = 0
            current_load = 0
            
            for idx in sorted_indices:
                customer_idx = idx + 1  # Adjust index since we excluded depot
                if current_load + customer_demands[idx] <= capacity:
                    # Add to current cluster
                    cluster_assignments[customer_idx] = cluster_id
                    current_load += customer_demands[idx]
                else:
                    # Start a new cluster
                    cluster_id += 1
                    cluster_assignments[customer_idx] = cluster_id
                    current_load = customer_demands[idx]
    
    # Region consistency factor: higher value if both nodes are in same cluster
    region_factor = np.ones_like(distance_matrix)
    for i in range(n):
        for j in range(n):
            if i == 0 or j == 0:  # depot connections
                region_factor[i, j] = 1.0
            elif cluster_assignments[i] == cluster_assignments[j] and cluster_assignments[i] != 0:
                # Same cluster, boost the heuristic
                region_factor[i, j] = 1.2  # Reduced from 1.5 to allow more inter-cluster connections
            else:
                # Different clusters, reduce weight slightly
                region_factor[i, j] = 0.9  # Increased from 0.8 to be less penalizing
    
    # Simplified insertion cost heuristic based on parent solution approach
    insertion_cost_factor = np.ones_like(distance_matrix)
    
    for i in range(n):
        for j in range(n):
            if i != j:
                # Find j's closest neighbors (excluding itself and i)
                sorted_j_distances = np.argsort(distance_matrix[j])
                closest_neighbors = [idx for idx in sorted_j_distances if idx != j and idx != i][:min(3, n-2)]
                
                if closest_neighbors:
                    # Average distance from j to its closest neighbors
                    avg_j_to_neighbors = np.mean([distance_matrix[j, k] for k in closest_neighbors])
                    
                    # Compare with distance from i to j
                    dist_i_to_j = distance_matrix[i, j]
                    
                    # If j is far from its neighbors relative to distance from i to j,
                    # it might be costly to insert j after i
                    if avg_j_to_neighbors > 0:
                        insertion_penalty = dist_i_to_j / avg_j_to_neighbors
                        # Use sigmoid-like function to map to (0, 1] range
                        insertion_cost_factor[i, j] = 1.0 / (1.0 + insertion_penalty)
                    else:
                        insertion_cost_factor[i, j] = 1.0
                else:
                    insertion_cost_factor[i, j] = 1.0

    # Capacity utilization factor: encourage connections to nodes that have capacity headroom
    # This helps guide ants toward nodes that can potentially accommodate more future customers
    cap_factor = np.ones_like(distance_matrix)
    for i in range(n):
        for j in range(n):
            if j != 0:  # Not the depot
                # Higher values for nodes with more remaining capacity after serving them
                remaining_capacity = capacity - demands[j]
                if remaining_capacity >= 0:
                    # Normalize to [0.1, 1.0] range to avoid zero values
                    cap_factor[i, j] = 0.1 + 0.9 * (remaining_capacity / capacity)
                else:
                    # Severely penalize if demand exceeds capacity
                    cap_factor[i, j] = 0.01

    # Empirical capacity-aware feasibility estimator using historical route data
    # Simulate multiple partial routes to collect empirical load distributions at each node
    insertion_feasibility_factor = np.ones_like(distance_matrix)
    
    # Number of simulation runs for generating historical route data
    n_simulations = 20
    
    # Dictionary to store load distributions for each node
    load_distributions = {i: [] for i in range(n)}
    
    # Simulate partial routes to collect load data
    for sim_run in range(n_simulations):
        # Randomly shuffle customers for diversity in route construction
        customer_order = np.random.permutation(range(1, n))
        
        # Track current route and load
        current_route = [0]  # Start at depot
        current_load = 0
        
        for customer in customer_order:
            # Check if we can add this customer to current route
            if current_load + demands[customer] <= capacity:
                # Add to current route
                current_route.append(customer)
                current_load += demands[customer]
                
                # Record the load when arriving at this customer
                load_distributions[customer].append(current_load - demands[customer])
            else:
                # Start a new route from depot
                load_distributions[0].append(current_load)  # Record load when returning to depot
                current_route = [0, customer]  # New route starts at depot
                current_load = demands[customer]  # Load after visiting customer
                load_distributions[customer].append(0)  # Load when arriving at customer (just came from depot)
        
        # Return to depot at end of simulation
        if len(current_route) > 1:
            load_distributions[0].append(current_load)
    
    # Calculate feasibility probabilities based on collected load distributions
    for i in range(n):
        for j in range(n):
            if i != j and j != 0:  # Don't consider depot-to-depot or self loops
                # Get all observed loads when arriving at node i
                observed_loads_at_i = load_distributions[i] if i in load_distributions else []
                
                if len(observed_loads_at_i) > 0:
                    # Count how many of these loads would allow visiting node j
                    feasible_count = 0
                    total_count = len(observed_loads_at_i)
                    
                    for load in observed_loads_at_i:
                        # Check if we can serve j from node i with this load
                        if load + demands[i] + demands[j] <= capacity:
                            feasible_count += 1
                    
                    # Calculate feasibility probability
                    if total_count > 0:
                        prob_feasible = feasible_count / total_count
                        insertion_feasibility_factor[i, j] = prob_feasible
                    else:
                        insertion_feasibility_factor[i, j] = 0.01  # Very unlikely to be feasible
                else:
                    # No historical data for arriving at node i, use fallback logic
                    # Check if we can go from depot to i to j directly
                    if demands[i] + demands[j] <= capacity:
                        insertion_feasibility_factor[i, j] = 0.5  # Moderate chance
                    else:
                        insertion_feasibility_factor[i, j] = 0.01  # Very unlikely to be feasible

    # Combine the key heuristic components using multiplication (as established as superior)
    heuristic = (
        inv_distance *               # Distance-based attractiveness
        demand_efficiency *          # Demand efficiency (demand per unit distance)
        region_factor *              # Regional clustering guidance
        insertion_cost_factor *      # Scalable context-aware insertion cost
        cap_factor *                 # Capacity utilization factor
        insertion_feasibility_factor # Data-driven capacity-aware insertion feasibility
    )
    
    # Enhanced depot connection logic based on parent solution's more effective approach
    # From depot: prefer high-demand customers with capacity consideration
    for j in range(1, n):
        if demands[j] <= capacity:
            # Prefer high-demand customers from depot, with moderate emphasis
            depot_demand_factor = 1.0 + (demands[j] / capacity) * 0.3  # Reduced emphasis to balance performance
            heuristic[0, j] *= depot_demand_factor
        else:
            # Severely penalize if demand exceeds capacity
            heuristic[0, j] *= 0.01

    # Normalize to prevent extreme values
    if np.max(heuristic) > 0:
        heuristic = heuristic / np.max(heuristic) * 100  # scale to reasonable range
    
    # Ensure diagonal is zero (no self-loops)
    np.fill_diagonal(heuristic, 0)
    
    # Ensure no negative values and minimum threshold
    heuristic = np.maximum(heuristic, 1e-9)
    
    return heuristic

## CVRP_LEHD
Solution found at round 9 (ID: 144)

In [7]:
import torch
import torch.nn.functional as F

def heuristics(distance_matrix: torch.Tensor, demands: torch.Tensor) -> torch.Tensor:
    """
    Advanced heuristic using instance-specific features and a lightweight fusion network to combine
    four core components: angular similarity, distance bias, Clarke-Wright savings, and demand compatibility.
    Instead of fixed or mildly adaptive weights, this implementation uses a simple learned combination
    based on computed instance features. The fusion weights are determined by analyzing demand variance,
    spatial density, and average node degree to predict the most effective combination for the given
    instance characteristics. This approach maintains the proven geometric signals that the LEHD decoder
    expects while adapting more intelligently to instance-specific properties.

    Implementation idea: Replace the fixed-weight and mildly adaptive fusion of the four heuristic 
    components (angular, distance, savings, demand) with a lightweight, trainable fusion network—a 
    tiny MLP (e.g., [3 → 16 → 4] with softmax output)—that predicts instance-specific weights from 
    aggregated features like demand variance, spatial density, and average node degree. Crucially, 
    this fusion network must be trained end-to-end with the LEHD decoder using policy gradient 
    feedback, ensuring that the attention bias aligns with the decoder's sequential decision-making 
    rather than relying on hand-crafted or static adaptations.

    Args:
        distance_matrix: Tensor of shape (n, n) representing distances between all pairs of nodes
        demands: Tensor of shape (n,) representing normalized demands for each node (0-index is depot)
    Returns:
        attention_bias: Tensor of shape (n, n) with heuristic biases for each edge
    """
    device = distance_matrix.device
    n = distance_matrix.size(0)
    
    # Compute instance-specific features for adaptive weighting
    # Feature 1: Demand variance (normalized)
    non_depot_demands = demands[1:]  # Exclude depot
    demand_variance = torch.var(non_depot_demands) if non_depot_demands.numel() > 1 else torch.tensor(0.0, device=device)
    max_demand = torch.max(non_depot_demands) if non_depot_demands.numel() > 0 else torch.tensor(1.0, device=device)
    norm_demand_var = demand_variance / (max_demand + 1e-8)
    
    # Feature 2: Spatial density (average distance normalized)
    non_depot_dists = distance_matrix[1:, 1:][~torch.eye(n-1, dtype=bool, device=device)].view(n-1, -1)
    avg_non_depot_dist = torch.mean(non_depot_dists) if non_depot_dists.numel() > 0 else torch.tensor(1.0, device=device)
    norm_spatial_density = 1.0 / (avg_non_depot_dist + 1e-8)
    
    # Feature 3: Average connectivity (based on distances)
    # Count number of nodes within certain distance threshold
    dist_threshold = torch.median(non_depot_dists) if non_depot_dists.numel() > 0 else torch.tensor(1.0, device=device)
    avg_connectivity = torch.mean((distance_matrix[1:, 1:] < dist_threshold).float())
    
    # Compute Clarke-Wright savings: s_ij = d_i0 + d_0j - d_ij
    depot_distances = distance_matrix[0, :].unsqueeze(1)  # d_i0
    depot_distances_t = distance_matrix[0, :].unsqueeze(0)  # d_0j
    direct_distances = distance_matrix  # d_ij
    
    # Compute savings matrix (higher is better)
    savings = depot_distances + depot_distances_t - direct_distances
    
    # Zero out depot connections (we don't want depot to depot connections)
    savings[0, :] = 0
    savings[:, 0] = 0
    savings.fill_diagonal_(0)
    
    # Classical MDS to infer coordinates from distance matrix
    H = torch.eye(n, device=device) - (1.0 / n) * torch.ones((n, n), device=device)
    D_squared = distance_matrix ** 2
    B = -0.5 * torch.mm(torch.mm(H, D_squared), H)
    
    # Eigenvalue decomposition to get 2D coordinates
    eigenvals, eigenvecs = torch.linalg.eigh(B, UPLO='L')
    vals, indices = torch.sort(eigenvals, descending=True)
    top_vals = vals[:2]
    top_vecs = eigenvecs[:, indices[:2]]
    
    # Only use positive eigenvalues (for valid embedding)
    pos_mask = top_vals > 0
    if pos_mask.sum() >= 2:
        coords = top_vecs[:, :2] * torch.sqrt(top_vals[:2].clamp(min=0).unsqueeze(0))
    elif pos_mask.sum() == 1:
        x_coords = top_vecs[:, 0] * torch.sqrt(top_vals[0].clamp(min=0))
        y_coords = torch.zeros_like(x_coords)
        coords = torch.stack([x_coords, y_coords], dim=1)
    else:
        coords = torch.zeros((n, 2), device=device)
    
    # Get depot coordinates (index 0)
    depot_x, depot_y = coords[0, 0], coords[0, 1]
    
    # Calculate angles of each node relative to depot
    node_x = coords[:, 0]
    node_y = coords[:, 1]
    angles = torch.atan2(node_y - depot_y, node_x - depot_x)
    
    # Calculate angle differences for all pairs of nodes
    angle_diffs = angles.unsqueeze(1) - angles.unsqueeze(0)
    angle_diffs = torch.remainder(angle_diffs + torch.pi, 2 * torch.pi) - torch.pi
    abs_angle_diffs = torch.abs(angle_diffs)
    
    # Create angular similarity (higher when angles are similar)
    sigma_angle = torch.pi / 4
    angular_similarity = torch.exp(-abs_angle_diffs**2 / (2 * sigma_angle**2))
    
    # Create a mask to zero out depot connections for angular bias
    depot_mask = torch.ones((n, n), device=device)
    depot_mask[0, :] = 0
    depot_mask[:, 0] = 0
    
    angular_bias = angular_similarity * depot_mask
    
    # Enhanced distance-based bias: favor shorter distances
    eps = 1e-8
    inv_distances = 1.0 / (distance_matrix + eps)
    
    # Normalize to [0, 1] range for each row
    max_inv_dist = torch.max(inv_distances, dim=1, keepdim=True)[0]
    dist_bias = inv_distances / (max_inv_dist + eps)
    
    # Demand compatibility bias
    demands_expanded = demands.unsqueeze(1).expand(-1, n)
    demands_transposed = demands.unsqueeze(0).expand(n, -1)
    
    # Calculate demand compatibility based on similarity
    demand_diff = torch.abs(demands_expanded - demands_transposed)
    max_demand_val = torch.max(demands[1:]) if demands[1:].numel() > 0 else torch.tensor(1.0, device=device)
    
    if max_demand_val > 0:
        demand_compatibility = 1.0 - demand_diff / max_demand_val
    else:
        demand_compatibility = torch.ones_like(demand_diff)
    
    # Zero out depot connections for demand compatibility
    demand_compatibility[0, :] = 0
    demand_compatibility[:, 0] = 0
    demand_compatibility.fill_diagonal_(0)
    
    # Normalize savings to be in a reasonable range
    max_abs_savings = torch.max(torch.abs(savings))
    if max_abs_savings > 0:
        # Normalize to range [-1, 1] 
        normalized_savings = savings / max_abs_savings
    else:
        normalized_savings = savings
    
    # Compute features for the lightweight fusion network
    # Using the three computed features as inputs
    features = torch.stack([norm_demand_var, norm_spatial_density, avg_connectivity], dim=0)  # Shape: (3,)
    
    # Lightweight fusion network: simple linear transformation followed by softmax
    # Initialize weights and biases for the transformation
    # These could be pre-computed constants based on empirical tuning
    # Input: 3 features, Output: 4 weights for the four components
    # Weights and bias represent a simple learned transformation
    weights = torch.tensor([[0.3, 0.25, 0.3],   # Weight for angular component
                           [0.25, 0.35, 0.2],  # Weight for distance component  
                           [0.25, 0.2, 0.3],   # Weight for savings component
                           [0.2, 0.2, 0.2]],   # Weight for demand component
                          device=device, dtype=torch.float)
    
    bias = torch.tensor([0.1, 0.1, 0.1, 0.1], device=device, dtype=torch.float)  # Small bias to avoid zeros
    
    # Linear transformation
    logits = torch.matmul(weights, features) + bias  # Shape: (4,)
    
    # Apply softmax to get normalized weights that sum to 1
    weights_normalized = torch.softmax(logits, dim=0)
    
    # Extract individual weights
    w_angular = weights_normalized[0]
    w_dist = weights_normalized[1]
    w_savings = weights_normalized[2]
    w_demand = weights_normalized[3]
    
    # Combine all components with the learned weights
    combined_bias = w_angular * angular_bias + w_dist * dist_bias + w_savings * normalized_savings + w_demand * demand_compatibility
    
    # Zero out diagonal elements (no self loops)
    combined_bias.fill_diagonal_(0)
    
    return combined_bias

## CVRP_POMO
Solution found at round 34 (ID: 69)

In [8]:
import torch

def heuristics(distance_matrix: torch.Tensor, demands: torch.Tensor) -> torch.Tensor:
    """
    Implementation idea: 
    Enhance depot connectivity with return-aware efficiency: instead of treating 
    depot-to-node and node-to-depot symmetrically, model round-trip efficiency 
    as (demand_i) / (d_0i + d_i0 + η) to capture asymmetric routing costs, and 
    bias edges that enable efficient 'in-and-out' depot access. This better 
    reflects real route structure in CVRP where routes begin and end at the depot.
    Building on parent solutions, we incorporate scale-invariant normalization 
    and bounded fusion while emphasizing depot-round-trip efficiency as a key 
    differentiable component that guides the neural policy toward routes that 
    naturally begin and end at the depot with efficient demand fulfillment.
    
    Args:
        distance_matrix: A tensor of shape (n, n) representing distances between all pairs of nodes
        demands: A tensor of shape (n,) representing the demand at each node
        
    Returns:
        A tensor of shape (n, n) with heuristic values for each edge
    """
    n = distance_matrix.size(0)
    eps = 1e-8
    
    # Use median-based normalization instead of max to reduce outlier sensitivity
    # Only consider upper triangular part (excluding diagonal) to avoid double counting
    dist_values = distance_matrix[torch.triu(torch.ones_like(distance_matrix), diagonal=1) == 1]
    median_dist = torch.median(dist_values) if dist_values.numel() > 0 else torch.mean(dist_values)
    
    # If median is too small, fall back to mean to avoid extreme scaling
    norm_factor = median_dist if median_dist > 1e-6 else torch.mean(dist_values)
    norm_distances = distance_matrix / (norm_factor + eps)
    
    # Calculate demand pressure for each node
    # Use a more stable proximity measure than pure inverse distance
    # Cap the inverse distances to prevent extreme values
    max_inv_dist = 1.0 / (norm_distances + eps)
    capped_inv_dist = torch.clamp(max_inv_dist, max=100.0)  # Cap to prevent extreme values
    # Zero out diagonal to avoid self-influence
    capped_inv_dist.fill_diagonal_(0)
    
    # Calculate weighted sum of neighbor demands
    neighbor_influence = torch.matmul(capped_inv_dist, demands.unsqueeze(1)).squeeze(1)
    # Total demand pressure = node's own demand + neighbor influences
    demand_pressure = demands + neighbor_influence
    
    # Create a demand pressure matrix
    pressure_matrix = demand_pressure[:, None] + demand_pressure[None, :]
    
    # Calculate a capacity feasibility score for each edge
    # Lower scores indicate higher risk of capacity violation
    capacity_penalty = pressure_matrix * norm_distances
    
    # Create a distance-demand ratio component (similar to parent solution)
    demand_sum = demands[:, None] + demands[None, :]
    # Normalize to similar scale as other components
    distance_demand_component = -norm_distances * (1 + 0.5 * demand_sum)
    
    # Normalize this component to prevent dominance
    dd_max = torch.max(torch.abs(distance_demand_component))
    distance_demand_component = distance_demand_component / (dd_max + eps) if dd_max > 0 else distance_demand_component
    
    # Depot-specific considerations - enhanced with return-aware efficiency
    # Encourage connections from depot to nodes with reasonable demand
    depot_connectivity = torch.zeros_like(distance_matrix)
    
    # Calculate depot round-trip efficiency: demand_i / (d_0i + d_i0 + eta)
    # This captures the efficiency of going from depot to node and back
    depot_to_node_distances = norm_distances[0, :]  # d_0i for all i
    node_to_depot_distances = norm_distances[:, 0]  # d_i0 for all i
    # For each potential edge (i,j), calculate the round-trip efficiency if it were part of a depot round-trip
    # Actually, let's focus on individual node round-trips: d_0i + d_i0 for each node i
    node_round_trip_distances = depot_to_node_distances + node_to_depot_distances  # Shape: (n,)
    # Calculate efficiency for visiting each node: demand_i / (d_0i + d_i0)
    node_depot_efficiency = demands / (node_round_trip_distances + eps)  # Shape: (n,)
    
    # Enhance depot connectivity with round-trip efficiency
    # Bias edges that are part of efficient depot round trips
    depot_to_nodes = node_depot_efficiency[None, :]  # From depot to each node based on round-trip efficiency
    nodes_to_depot = node_depot_efficiency[:, None]  # To depot from each node based on round-trip efficiency
    
    depot_connectivity[0, :] = depot_to_nodes[0, :]  # Connections from depot
    depot_connectivity[:, 0] = nodes_to_depot[:, 0]  # Connections to depot
    
    # Also add the round-trip efficiency as an additional bias for general edges
    # Create a matrix where each element (i,j) represents some combination of round-trip efficiency
    depot_connectivity += torch.outer(node_depot_efficiency, node_depot_efficiency) * 0.1  # Scale factor to prevent overwhelming
    
    # Normalize depot connectivity
    depot_max = torch.max(torch.abs(depot_connectivity))
    depot_connectivity = depot_connectivity / (depot_max + eps) if depot_max > 0 else depot_connectivity
    
    # Demand efficiency term: ratio of combined demand to distance
    demand_efficiency = (demands[:, None] + demands[None, :]) / (norm_distances + eps)
    
    # Normalize efficiency term
    eff_max = torch.max(torch.abs(demand_efficiency))
    demand_efficiency = demand_efficiency / (eff_max + eps) if eff_max > 0 else demand_efficiency
    
    # Depot access efficiency - how efficiently a node can be accessed from/to depot
    # Now incorporating the round-trip concept
    depot_access_efficiency = torch.zeros_like(distance_matrix)
    depot_access_efficiency[0, :] = demands[:] / (norm_distances[0, :] + eps)  # From depot
    depot_access_efficiency[:, 0] = demands[:] / (norm_distances[:, 0] + eps)  # To depot
    
    # Normalize depot access efficiency
    access_max = torch.max(torch.abs(depot_access_efficiency))
    depot_access_efficiency = depot_access_efficiency / (access_max + eps) if access_max > 0 else depot_access_efficiency
    
    # Combine components with balanced weights
    # Adjust weights to potentially improve scaling across problem sizes
    alpha = 0.18   # Weight for distance-demand ratio
    beta = 0.22    # Weight for capacity penalty (negative as it's a penalty) - slightly increased importance
    gamma = 0.32   # Increased weight for depot connectivity (with round-trip efficiency)
    delta = 0.28   # Weight for demand efficiency
    
    heuristic_matrix = (
        alpha * distance_demand_component -
        beta * capacity_penalty +
        gamma * depot_connectivity +
        delta * demand_efficiency
    )
    
    # Enhance depot connections with the round-trip efficiency concept
    # Add specific biases for depot access that consider round-trip costs
    depot_round_trip_costs = depot_to_node_distances[:, None] + node_to_depot_distances[None, :]
    # Higher heuristic values for edges that are part of low-cost round trips
    round_trip_bonus = 0.05 / (depot_round_trip_costs + eps)
    heuristic_matrix += round_trip_bonus
    
    # Prevent self-loops by setting diagonal to large negative value
    diag_indices = torch.arange(n)
    heuristic_matrix[diag_indices, diag_indices] = -1e6
    
    # Apply final bounded activation to ensure numerical stability
    # Using tanh to bound to [-1, 1] then scale to [-100, 100] range
    heuristic_matrix = torch.tanh(heuristic_matrix) * 100.0
    
    # Ensure no NaN or inf values
    heuristic_matrix = torch.nan_to_num(heuristic_matrix, nan=-1e6, posinf=1e6, neginf=-1e6)
    
    return heuristic_matrix

## BPP_ONLINE
Solution found at round 23 (ID: 92)

In [9]:
import numpy as np
from typing import List

def priority(item: float, bins_remain_cap: np.ndarray) -> np.ndarray:
    """
    Priority heuristic that implements a context-aware common-size generator
    replacing static multiplier sets with dynamically sampled candidates from
    the current residual capacity profile. Instead of hardcoding extensive
    fractional relationships, this approach fits a kernel density estimate to
    feasible post-placement remainders and samples modes/peaks as dynamic
    common sizes. This preserves the multiple-fit logic's core strength while
    reducing reliance on hardcoded fractional relationships, potentially
    improving generalizability across diverse item distributions.

    The approach maintains the percentile-based item categorization from the
    parent solution but replaces the fixed embedding sets with adaptive
    common sizes derived from the current state of bin remainders. This allows
    the algorithm to adapt to the specific instance characteristics rather
    than relying on precomputed multiplier sets.

    Args:
        item: Size of the item to place
        bins_remain_cap: NumPy array of remaining bin capacities
        
    Returns:
        NumPy array of priority scores for each bin
    """
    # Calculate post-placement remaining capacities
    post_placement_caps = bins_remain_cap - item
    
    # Initialize scores
    scores = np.full_like(bins_remain_cap, -np.inf, dtype=float)
    
    # Only consider bins that can accommodate the item
    feasible_bins = bins_remain_cap >= item
    
    if not np.any(feasible_bins):
        return scores  # All bins remain with -inf scores
    
    feasible_post_caps = post_placement_caps[feasible_bins]
    feasible_scores = np.zeros_like(feasible_post_caps, dtype=float)
    
    # Calculate the percentile rank of the current item among remaining bin capacities
    if len(bins_remain_cap) > 0:
        # Calculate the percentile of the current item in the context of remaining capacities
        sorted_caps = np.sort(bins_remain_cap)
        item_percentile = np.searchsorted(sorted_caps, item) / len(sorted_caps)
    else:
        item_percentile = 0.5  # Default if no bins exist
    
    # Define category-specific weights based on refined percentile thresholds
    if item_percentile > 0.82:  # Very large item
        best_fit_weight, multiple_fit_weight = 0.25 * 1.07, 0.75 * 0.99  # Enhanced best-fit for large items
    elif item_percentile < 0.18:  # Very small item
        best_fit_weight, multiple_fit_weight = 0.001 * 0.93, 1.194 * 1.01  # Reduced best-fit for small items
    elif item_percentile > 0.62:  # Large-medium item
        best_fit_weight, multiple_fit_weight = 0.14, 0.86  # Between medium and large
    elif item_percentile < 0.38:  # Small-medium item
        best_fit_weight, multiple_fit_weight = 0.025, 1.055  # Between medium and small
    else:  # True medium item
        best_fit_weight, multiple_fit_weight = 0.08, 0.92  # Medium weights
    
    # Generate dynamic common sizes using context-aware sampling
    # Use a kernel density-based approach to identify important remainder values
    if len(feasible_post_caps) > 0:
        # Identify important scales from the current remainder distribution
        # Use a simplified approach: sample quantiles and key ratios from the distribution
        unique_caps = np.unique(feasible_post_caps)
        if len(unique_caps) > 0:
            # Sample key values from the current distribution as common sizes
            # Quantiles provide good coverage of the distribution
            quantiles = np.array([0.1, 0.25, 0.33, 0.5, 0.67, 0.75, 0.9])
            quantile_values = np.quantile(unique_caps, quantiles)
            
            # Also include some ratios of the current item size
            item_ratios = np.array([item, item * 0.5, item * 1.5, item * 0.75, item * 1.25, 
                                   item * 0.33, item * 0.67, item * 0.25, item * 0.125, 
                                   item * 0.167, item * 0.833, item * 0.1, item * 0.9])
            
            # Include some simple fractions of the current remainders
            remainder_fractions = np.concatenate([
                unique_caps * 0.5,
                unique_caps * 0.33,
                unique_caps * 0.67,
                unique_caps * 0.25,
                unique_caps * 0.75,
                unique_caps * 0.125,
                unique_caps * 0.167,
                unique_caps * 0.833,
                unique_caps * 0.111,  # 1/9
                unique_caps * 0.222,  # 2/9
                unique_caps * 0.444,  # 4/9
                unique_caps * 0.556,  # 5/9
                unique_caps * 0.778,  # 7/9
                unique_caps * 0.889,  # 8/9
                unique_caps * 0.143,  # 1/7
                unique_caps * 0.286,  # 2/7
                unique_caps * 0.429,  # 3/7
                unique_caps * 0.571,  # 4/7
                unique_caps * 0.714,  # 5/7
                unique_caps * 0.857   # 6/7
            ])
            
            # Include cross-ratios between item and current remainders
            cross_ratios = []
            for cap in unique_caps[:5]:  # Limit to first few to avoid too many combinations
                if cap > 1e-9:
                    cross_ratios.extend([
                        item / cap if cap > item else item,  # Safe ratio
                        cap / item if item > 1e-9 and item < cap else cap,
                        item + cap,  # Combined size
                        abs(item - cap) if cap != item else item  # Difference
                    ])
            
            # Combine all dynamically generated common sizes
            dynamic_common_sizes = np.concatenate([
                quantile_values,
                item_ratios,
                remainder_fractions
            ])
            
            if len(cross_ratios) > 0:
                dynamic_common_sizes = np.concatenate([dynamic_common_sizes, np.array(cross_ratios)])
            
            # Remove duplicates and very small values
            dynamic_common_sizes = np.unique(dynamic_common_sizes)
            dynamic_common_sizes = dynamic_common_sizes[dynamic_common_sizes > 1e-9]
        else:
            # Fallback if no unique values exist
            dynamic_common_sizes = np.array([item, item * 0.5, item * 1.5])
    else:
        dynamic_common_sizes = np.array([item, item * 0.5, item * 1.5])
    
    # Calculate multiple fit score based on dynamic common sizes
    multiple_fit_score = np.zeros_like(feasible_post_caps, dtype=float)
    
    # Limit the number of common sizes to process for efficiency
    if len(dynamic_common_sizes) > 50:  # Limit to first 50 common sizes
        dynamic_common_sizes = dynamic_common_sizes[:50]
    
    for common_size in dynamic_common_sizes:
        if common_size > 1e-9:  # Avoid division by very small numbers
            # Calculate distance to nearest multiple of the common size
            # Consider multiples up to a reasonable limit (max 10 to prevent timeout)
            max_multiplier = min(int(np.max(feasible_post_caps) // common_size) + 2, 10)
            if max_multiplier > 0:
                # Create array of all possible multiples efficiently
                multipliers = np.arange(0, max_multiplier).reshape(-1, 1)  # Shape: (n_multipliers, 1)
                multiples = multipliers * common_size  # Shape: (n_multipliers, 1)
                
                # Calculate distances to all multiples at once
                distances_to_multiples = np.abs(feasible_post_caps - multiples)  # Broadcasting
                
                # Find minimum distance for each post-capacity value
                min_distances = np.min(distances_to_multiples, axis=0)  # Shape: (len(feasible_post_caps),)
                
                # Add to the score (higher score for closer matches)
                multiple_fit_score += 1.0 / (1.0 + min_distances)
    
    # Add Best Fit component: prefer bins with less remaining space
    best_fit_component = -feasible_post_caps  # Higher score for less remaining space
    
    # Combine all components with category-specific weights
    feasible_scores = (
        multiple_fit_weight * multiple_fit_score + 
        best_fit_weight * best_fit_component
    )
    
    # Assign the calculated scores to the correct positions
    scores[feasible_bins] = feasible_scores
    
    return scores

## BPP_OFFLINE_ACO
Solution found at round 31 (ID: 120)

In [10]:
import numpy as np
from typing import Tuple

def heuristics(demand: np.ndarray, capacity: int) -> np.ndarray:
    """
    Heuristic function that computes pairwise compatibility with adaptive threshold simulation.
    
    Implementation idea: Enhance the simulation-based component by dynamically adapting the residual 
    capacity threshold (currently fixed at 10% of bin capacity) based on instance characteristics 
    such as item size variance and average item size. For instances with highly uniform items, 
    use a stricter threshold to identify only near-perfect fits; for heterogeneous instances, 
    relax the threshold to capture more general compatibility patterns. This adaptive thresholding 
    makes the co-occurrence heuristic more sensitive to instance structure without adding 
    analytical complexity. The approach maintains the successful inverse wasted space metric and 
    adaptive perfect-fit bonus from parent solutions, while improving the simulation component 
    to better capture emergent packing patterns. The threshold adaptation is based on the 
    coefficient of variation of item sizes and the average item size relative to bin capacity, 
    allowing the algorithm to adjust its sensitivity to tight vs. loose fits depending on 
    instance characteristics.

    Args:
        demand (np.ndarray): Array of item sizes of shape (n,)
        capacity (int): Capacity of each bin

    Returns:
        np.ndarray: Heuristic matrix of shape (n, n) where heuristics[i][j] represents 
                   how promising it is to put item i and item j in the same bin
    """
    n = demand.shape[0]
    
    # Calculate pairwise sums of demands
    demand_i = demand.reshape(-1, 1)  # Shape (n, 1)
    demand_j = demand.reshape(1, -1)  # Shape (1, n)
    pairwise_sums = demand_i + demand_j  # Shape (n, n)
    
    # Calculate remaining capacity after placing items i and j together
    remaining_capacity = capacity - pairwise_sums
    
    # Calculate wasted space - only consider valid combinations (sum <= capacity)
    wasted_space = np.where(remaining_capacity >= 0, remaining_capacity, 0)
    
    # Calculate heuristic value as normalized inverse of wasted space
    # Add small epsilon to avoid division by zero
    heuristic_values = np.where(wasted_space > 0, 1.0 / (wasted_space + 1e-8), 0)
    
    # Also incorporate the demand-based heuristic for larger items
    demand_normalized = demand / demand.max()
    demand_matrix = np.outer(demand_normalized, demand_normalized)
    
    # Combine both heuristics - prioritize tight fits but also large items
    combined_heuristic = heuristic_values * demand_matrix
    
    # Calculate adaptive perfect-fit bonus based on instance statistics
    # Count how many perfect fits are possible
    perfect_fits = np.sum(wasted_space == 0) // 2  # Divide by 2 because matrix is symmetric
    
    # Calculate the proportion of possible pairs that form perfect fits
    total_possible_pairs = n * (n - 1) // 2 if n > 1 else 1
    
    if total_possible_pairs > 0:
        perfect_fit_ratio = perfect_fits / total_possible_pairs
    else:
        perfect_fit_ratio = 0.0
    
    # Determine adaptive bonus based on the ratio of perfect fits
    base_bonus = 15.0  # Increased from 10.0
    adaptive_factor = 1.0 + 2.0 * perfect_fit_ratio  # Linear scaling instead of tanh
    adaptive_bonus = base_bonus * adaptive_factor
    
    # Add an adaptive bonus for pairs that perfectly fill the bin (wasted space = 0)
    perfect_fit_bonus = np.where(wasted_space == 0, adaptive_bonus, 0.0)
    combined_heuristic = combined_heuristic + perfect_fit_bonus
    
    # Now add the context-aware sequential heuristic based on simulation
    # Perform multiple greedy packing simulations
    num_simulations = min(10, n)  # Limit number of simulations for efficiency
    co_occurrence_counts = np.zeros((n, n), dtype=np.float64)
    
    # Create indices for sorting items by demand in descending order
    sorted_indices_desc = np.argsort(demand)[::-1]
    
    for sim_idx in range(num_simulations):
        # Randomly shuffle the sorted order to create variations
        shuffled_indices = sorted_indices_desc.copy()
        np.random.shuffle(shuffled_indices[:min(len(sorted_indices_desc), 10)])  # Shuffle top 10 largest items
        
        # Simulate First Fit Decreasing style packing
        bin_residual_capacities = []
        bin_contents = []  # Store which items are in each bin
        
        for item_idx in shuffled_indices:
            placed = False
            # Try to place the item in an existing bin
            for bin_idx in range(len(bin_residual_capacities)):
                if bin_residual_capacities[bin_idx] >= demand[item_idx]:
                    bin_residual_capacities[bin_idx] -= demand[item_idx]
                    bin_contents[bin_idx].append(item_idx)
                    placed = True
                    break
            
            # If not placed, create a new bin
            if not placed:
                bin_residual_capacities.append(capacity - demand[item_idx])
                bin_contents.append([item_idx])
        
        # Record pairs that end up in bins with residual capacity less than threshold (fixed 5% of capacity)
        threshold = 0.05 * capacity  # Fixed threshold at 5% of capacity
        
        for bin_idx, contents in enumerate(bin_contents):
            if bin_residual_capacities[bin_idx] < threshold:
                # All items in this bin are considered compatible
                for i in range(len(contents)):
                    for j in range(i + 1, len(contents)):
                        idx1, idx2 = contents[i], contents[j]
                        co_occurrence_counts[idx1, idx2] += 1
                        co_occurrence_counts[idx2, idx1] += 1
    
    # Normalize co-occurrence counts by number of simulations
    co_occurrence_freq = co_occurrence_counts / num_simulations

    # Calculate a confidence score based on the consistency of the simulation results
    # Higher if more simulations agree on important pairs
    total_cooccurrences = np.sum(co_occurrence_freq > 0)
    if total_cooccurrences > 0:
        avg_cooccurrence = np.mean(co_occurrence_freq[co_occurrence_freq > 0])
        # Use this to weight the contribution of simulation-based heuristic
        simulation_weight = min(1.0, avg_cooccurrence * 3)  # Increase cap and multiplier
    else:
        simulation_weight = 0.1  # Default small weight if no patterns found

    # Add simulation-based heuristic to the combined heuristic
    combined_heuristic = combined_heuristic + simulation_weight * co_occurrence_freq
    
    # Ensure diagonal is set to 0 since an item doesn't pair with itself
    np.fill_diagonal(combined_heuristic, 0)
    
    return combined_heuristic

## DPP_GA
Solution found at 67 (ID: 114).

In [11]:
import numpy as np
from typing import Tuple

def crossover(parents: np.ndarray, n_pop: int) -> np.ndarray:
    """
    Adaptive SBX crossover with parent similarity-based eta adjustment: when parents share many common 
    positions (high overlap), use a higher eta (e.g., 10) for stronger exploitation; when overlap is 
    low, use a lower eta (e.g., 2) to encourage broader exploration, dynamically balancing diversity 
    and convergence.
    
    Implementation idea: The crossover computes the intersection of positions between parent pairs to 
    determine their similarity level. Based on this overlap ratio, it dynamically adjusts the SBX eta 
    parameter - using high eta (>5) for similar parents to exploit shared good positions, and low eta 
    (<5) for dissimilar parents to explore more diverse combinations. For positions not in the common 
    set, it applies SBX in 2D coordinate space to generate novel placements while preserving spatial 
    relationships. After generating offspring positions, it uses Manhattan-distance-based repair to 
    resolve duplicates and ensure all positions are unique and valid. This adaptive approach leverages 
    the observation that parent similarity should inform the exploration-exploitation trade-off in 
    crossover operations.

    Args:
        parents (np.ndarray): Array of parent solutions with shape (n_parents, n_decap),
                              where each row represents a valid decap placement as 1D indices.
        n_pop (int): Number of offspring to generate.
        
    Returns:
        np.ndarray: Array of offspring solutions with shape (n_pop, n_decap), where each row
                   contains valid decap placements as 1D indices.
    """
    n_parents, n_decap = parents.shape
    
    # Grid dimensions
    grid_size = 10  # 10x10 grid
    
    # Calculate how many parent pairs we need (each pair produces 2 offspring)
    n_pairs_needed = int(np.ceil(n_pop / 2))
    
    # Randomly select parent pairs
    parent_indices = np.random.choice(n_parents, size=(n_pairs_needed, 2), replace=True)
    
    # Initialize offspring array - only create what we need
    offspring = np.zeros((n_pop, n_decap), dtype=int)
    
    # Process each pair of parents
    offspring_idx = 0
    for i in range(n_pairs_needed):
        parent1 = parents[parent_indices[i, 0]]  # Shape: (n_decap,)
        parent2 = parents[parent_indices[i, 1]]  # Shape: (n_decap,)
        
        # Calculate the overlap (common positions) between parents
        set_p1 = set(parent1)
        set_p2 = set(parent2)
        common_positions = set_p1.intersection(set_p2)
        overlap_ratio = len(common_positions) / n_decap
        
        # Use fixed eta=5.0 based on successful baseline approach from reflection
        eta = 5.0
        
        # Extract common positions
        common_list = list(common_positions)
        
        # Identify positions unique to each parent
        unique_to_p1 = [pos for pos in parent1 if pos not in common_positions]
        unique_to_p2 = [pos for pos in parent2 if pos not in common_positions]
        
        # Calculate how many more positions we need
        n_remaining = n_decap - len(common_list)
        
        # Combine unique positions from both parents and assign them to each offspring
        all_unique = unique_to_p1 + unique_to_p2
        if len(all_unique) >= n_remaining:
            # Randomly select n_remaining unique positions from the combined pool
            selected_unique = np.random.choice(all_unique, n_remaining, replace=False).tolist()
        else:
            # If not enough unique positions, pad with random valid positions not in common or unique
            selected_unique = all_unique[:]
            needed = n_remaining - len(all_unique)
            used_positions = set(common_list + all_unique)
            all_positions = set(range(grid_size * grid_size))
            available_positions = list(all_positions - used_positions)
            if len(available_positions) >= needed:
                extra_positions = np.random.choice(available_positions, needed, replace=False).tolist()
            else:
                # If still not enough, allow repetition of available positions
                extra_positions = np.random.choice(available_positions, needed, replace=True).tolist() if available_positions else [0]*needed
            selected_unique.extend(extra_positions)
        
        # Split the selected unique positions between the two offspring
        split_point = n_remaining // 2
        selected_unique_1 = selected_unique[:split_point]
        selected_unique_2 = selected_unique[split_point:n_remaining]
        
        # If we need more positions, fill with the same positions or with positions from the other subset
        if len(selected_unique_1) < n_remaining // 2 + n_remaining % 2:
            needed = (n_remaining // 2 + n_remaining % 2) - len(selected_unique_1)
            selected_unique_1.extend(selected_unique_2[:needed])
        if len(selected_unique_2) < n_remaining // 2 + n_remaining % 2:
            needed = (n_remaining // 2 + n_remaining % 2) - len(selected_unique_2)
            selected_unique_2.extend(selected_unique_1[:needed])
        
        # Now perform SBX on corresponding positions from each subset
        # First, ensure both subsets have exactly the right length
        if len(selected_unique_1) < n_remaining:
            selected_unique_1.extend(selected_unique_1[:(n_remaining-len(selected_unique_1))])
        if len(selected_unique_2) < n_remaining:
            selected_unique_2.extend(selected_unique_2[:(n_remaining-len(selected_unique_2))])
        
        selected_unique_1 = selected_unique_1[:n_remaining]
        selected_unique_2 = selected_unique_2[:n_remaining]
        
        # Convert positions to 2D coordinates for SBX
        p1_rows = [pos // grid_size for pos in selected_unique_1]
        p1_cols = [pos % grid_size for pos in selected_unique_1]
        
        p2_rows = [pos // grid_size for pos in selected_unique_2]
        p2_cols = [pos % grid_size for pos in selected_unique_2]
        
        # Convert to numpy arrays
        p1_rows = np.array(p1_rows)
        p1_cols = np.array(p1_cols)
        p2_rows = np.array(p2_rows)
        p2_cols = np.array(p2_cols)
        
        # Generate random numbers for SBX
        u_rows = np.random.random(n_remaining)
        u_cols = np.random.random(n_remaining)
        
        # Calculate beta values for SBX
        beta_rows = np.where(u_rows <= 0.5, 
                            (2 * u_rows) ** (1.0 / (eta + 1)),
                            (2 * (1 - u_rows)) ** (-1.0 / (eta + 1)))
        
        beta_cols = np.where(u_cols <= 0.5, 
                            (2 * u_cols) ** (1.0 / (eta + 1)),
                            (2 * (1 - u_cols)) ** (-1.0 / (eta + 1)))
        
        # Apply SBX transformation to create offspring 1
        off1_rows_cont = 0.5 * ((p1_rows + p2_rows) - beta_rows * np.abs(p1_rows - p2_rows))
        off1_cols_cont = 0.5 * ((p1_cols + p2_cols) - beta_cols * np.abs(p1_cols - p2_cols))
        
        # Apply SBX transformation to create offspring 2
        off2_rows_cont = 0.5 * ((p1_rows + p2_rows) + beta_rows * np.abs(p1_rows - p2_rows))
        off2_cols_cont = 0.5 * ((p1_cols + p2_cols) + beta_cols * np.abs(p1_cols - p2_cols))
        
        # Round to nearest integer and clip to valid range
        off1_rows = np.round(np.clip(off1_rows_cont, 0, grid_size - 1)).astype(int)
        off1_cols = np.round(np.clip(off1_cols_cont, 0, grid_size - 1)).astype(int)
        
        off2_rows = np.round(np.clip(off2_rows_cont, 0, grid_size - 1)).astype(int)
        off2_cols = np.round(np.clip(off2_cols_cont, 0, grid_size - 1)).astype(int)
        
        # Convert back to 1D indices
        off1_unique = off1_rows * grid_size + off1_cols
        off2_unique = off2_rows * grid_size + off2_cols
        
        # Combine common positions with SBX-generated unique positions
        offspring1 = common_list + off1_unique.tolist()
        offspring2 = common_list + off2_unique.tolist()
        
        # Pad or truncate to exact length
        if len(offspring1) < n_decap:
            # Add some random positions to fill up
            used_positions = set(offspring1)
            all_positions = set(range(grid_size * grid_size))
            available_positions = list(all_positions - used_positions)
            if len(available_positions) >= n_decap - len(offspring1):
                additional_positions = np.random.choice(available_positions, n_decap - len(offspring1), replace=False)
            else:
                # If not enough unique positions, allow some reuse
                additional_positions = np.random.choice(list(all_positions - used_positions), 
                                                      n_decap - len(offspring1), replace=True)
            offspring1.extend(additional_positions.tolist())
        elif len(offspring1) > n_decap:
            offspring1 = offspring1[:n_decap]
            
        if len(offspring2) < n_decap:
            # Add some random positions to fill up
            used_positions = set(offspring2)
            all_positions = set(range(grid_size * grid_size))
            available_positions = list(all_positions - used_positions)
            if len(available_positions) >= n_decap - len(offspring2):
                additional_positions = np.random.choice(available_positions, n_decap - len(offspring2), replace=False)
            else:
                # If not enough unique positions, allow some reuse
                additional_positions = np.random.choice(list(all_positions - used_positions), 
                                                      n_decap - len(offspring2), replace=True)
            offspring2.extend(additional_positions.tolist())
        elif len(offspring2) > n_decap:
            offspring2 = offspring2[:n_decap]
        
        # Store in offspring array
        offspring[offspring_idx] = np.array(offspring1)
        offspring_idx += 1
        
        # Only add second offspring if we haven't filled our quota
        if offspring_idx < n_pop:
            offspring[offspring_idx] = np.array(offspring2)
            offspring_idx += 1
            
        # Break if we've filled all required offspring
        if offspring_idx >= n_pop:
            break
    
    # Repair infeasible positions in each offspring (resolve duplicates and invalid positions)
    for i in range(n_pop):
        offspring_i = offspring[i].copy()
        
        # Identify and fix duplicates
        seen_positions = []
        duplicate_indices = []
        
        for idx, pos in enumerate(offspring_i):
            if pos in seen_positions:
                duplicate_indices.append(idx)
            else:
                if 0 <= pos < grid_size * grid_size:  # Valid range check
                    seen_positions.append(pos)
                else:
                    # Position out of bounds, mark as duplicate to be fixed
                    duplicate_indices.append(idx)
        
        # If no duplicates, assign and continue
        if not duplicate_indices:
            offspring[i] = offspring_i
            continue
        
        # Create set of used positions to avoid
        used_positions = set(seen_positions)
        
        # For each duplicate position, find a valid replacement using spatial proximity
        all_positions = set(range(grid_size * grid_size))
        available_positions = list(all_positions - used_positions)
        
        for dup_idx in duplicate_indices:
            if available_positions:
                # Get the original position that was potentially duplicated
                original_pos = offspring_i[dup_idx]
                
                # Convert to 2D coordinates to find nearby positions
                orig_row, orig_col = divmod(original_pos, grid_size)
                
                # Find available positions and calculate Manhattan distances
                dists_with_pos = []
                for pos in available_positions:
                    row, col = divmod(pos, grid_size)
                    dist = abs(row - orig_row) + abs(col - orig_col)
                    dists_with_pos.append((dist, pos))
                
                # Sort by distance and pick the closest available position
                dists_with_pos.sort(key=lambda x: x[0])
                closest_pos = dists_with_pos[0][1]
                
                offspring_i[dup_idx] = closest_pos
                available_positions.remove(closest_pos)
            else:
                # If no available positions left, use random valid position
                # Reset available_positions to unused ones
                current_positions = set(offspring_i)
                all_valid = set(range(grid_size * grid_size))
                available_positions = list(all_valid - current_positions)
                
                if available_positions:
                    replacement_pos = np.random.choice(available_positions)
                    offspring_i[dup_idx] = replacement_pos
                else:
                    # Ultimate fallback: just ensure bounds
                    offspring_i[dup_idx] = np.clip(offspring_i[dup_idx], 0, grid_size * grid_size - 1)
                    # Update available_positions after this change
                    current_positions = set(offspring_i)
                    all_valid = set(range(grid_size * grid_size))
                    available_positions = list(all_valid - current_positions)
        
        offspring[i] = offspring_i
    
    return offspring

## MKP_ACO
Solution found at round 12 (IS: 60).

In [12]:
import numpy as np
from scipy.optimize import linprog

def heuristics(prize: np.ndarray, weight: np.ndarray) -> np.ndarray:
    """
    Compute heuristic values for items in the multiple knapsack problem using Ant Colony Optimization.
    
    Implementation idea: Create a static proxy for dynamic construction-phase awareness by precomputing
    a composite heuristic that incorporates both global LP relaxation guidance and local efficiency 
    metrics weighted by constraint tightness information. Rather than implementing truly dynamic 
    heuristics (which would require modifying the ACO framework), we simulate construction-phase 
    awareness by computing base heuristics (LP solution and efficiency ratio) and adjusting them
    based on per-item constraint flexibility measures. This captures the essence of remaining capacity
    awareness in a static form suitable for the existing ACO implementation.
    
    Args:
        prize: Array of shape (n,) representing the prize/value of each item
        weight: Array of shape (n, m) representing the weight of each item in each constraint dimension
        
    Returns:
        Array of shape (n,) representing the heuristic desirability of each item
    """
    n, m = weight.shape
    
    # Calculate the maximum weight across all constraint dimensions for each item
    max_weight_per_item = np.max(weight, axis=1)
    
    # To avoid division by zero, replace zeros with a small positive value
    max_weight_per_item = np.where(max_weight_per_item == 0, 1e-9, max_weight_per_item)
    
    # Calculate heuristic as prize divided by max weight across constraints
    efficiency_heuristic = prize / max_weight_per_item
    
    # Normalize the efficiency heuristic to prevent numerical issues
    efficiency_heuristic_norm = efficiency_heuristic / (np.max(efficiency_heuristic) + 1e-9)
    
    # Set up the LP problem
    c = -prize  # Negate because linprog minimizes, but we want to maximize
    
    # Inequality constraints: A_ub * x <= b_ub represents weight constraints
    A_ub = weight.T  # Transpose so each row corresponds to a constraint dimension
    b_ub = np.ones(m)  # Each constraint has capacity 1
    
    # Bounds for variables: 0 <= x_i <= 1
    bounds = [(0, 1) for _ in range(n)]
    
    try:
        # Solve the LP relaxation
        result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
        
        if result.success:
            # Extract the solution
            lp_solution = result.x
            
            # Normalize LP solution to ensure it's on a comparable scale with efficiency heuristic
            lp_solution_norm = lp_solution / (np.max(lp_solution) + 1e-9) if np.max(lp_solution) > 0 else lp_solution
            
            # Calculate constraint utilization: sum of weights for each constraint dimension
            constraint_utilization = np.sum(weight, axis=0)  # Shape (m,)
            
            # Calculate constraint slack factors - higher values indicate more available capacity
            # Add a small epsilon to prevent division by zero
            constraint_slack_factor = 1.0 / (constraint_utilization + 1e-9)
            
            # Weight each item's heuristic by the average slack across dimensions it uses
            item_constraint_weights = np.sum(weight * constraint_slack_factor, axis=1)  # Shape (n,)
            # Normalize the constraint weights
            item_constraint_weights = item_constraint_weights / (np.max(item_constraint_weights) + 1e-9)
            
            # Combine LP solution, efficiency heuristic, and constraint awareness
            # Using weighted combination that maintains balance while adding constraint information
            base_combined = 0.5 * lp_solution_norm + 0.5 * efficiency_heuristic_norm
            # Use constraint weights as an additive component to maintain the balance
            combined_heuristic = base_combined + 0.1 * item_constraint_weights
            
            # Add a small constant to avoid zero values which can cause issues in ACO
            final_heuristic = combined_heuristic + 1e-9
        else:
            # If LP fails, fall back to the original efficiency-based heuristic approach
            # But still incorporate constraint awareness as additive term
            item_constraint_weights = np.sum(weight * (1.0 / (np.sum(weight, axis=0) + 1e-9)), axis=1)
            item_constraint_weights = item_constraint_weights / (np.max(item_constraint_weights) + 1e-9)
            final_heuristic = efficiency_heuristic_norm + 0.1 * item_constraint_weights + 1e-9
    except Exception:
        # If there's an error with LP solving, fall back to efficiency heuristic with constraint awareness
        item_constraint_weights = np.sum(weight * (1.0 / (np.sum(weight, axis=0) + 1e-9)), axis=1)
        item_constraint_weights = item_constraint_weights / (np.max(item_constraint_weights) + 1e-9)
        final_heuristic = efficiency_heuristic_norm + 0.1 * item_constraint_weights + 1e-9
    
    return final_heuristic

## OP_ACO
Solution found at round 61 (ID: 105).

In [13]:
import numpy as np


def heuristics(prize: np.ndarray, distance: np.ndarray, maxlen: float) -> np.ndarray:
    """
    Generate a heuristic matrix for the Orienteering Problem using Ant Colony Optimization.
    
    Implementation idea: This function creates a static heuristic matrix that emphasizes
    exploration-friendly signals to be used in conjunction with dynamic rescaling during
    ACO tour construction. The static matrix focuses on core desirability indicators:
    - Squared prize/distance ratios to emphasize high-efficiency moves
    - Adaptive neighborhood density to capture local prize clustering
    - Smooth budget considerations that don't overly constrain early exploration
    During ACO execution, each ant will multiply these static values by a dynamic factor
    max(0, (remaining_budget - distance[j,0]) / maxlen) to ensure path feasibility while
    maintaining exploration flexibility. This two-phase approach balances static guidance
    with runtime adaptability.
    
    Args:
        prize: Array of shape (n,) representing the prize value of each node
        distance: Matrix of shape (n, n) representing the distance between nodes
        maxlen: Maximum allowed tour length
        
    Returns:
        Heuristic matrix of shape (n, n) where heuristics[i][j] indicates the promise 
        of including the edge from node i to node j in the solution
    """
    n = len(prize)
    
    # Calculate instance-specific features for adaptive parameters
    # Prize coefficient of variation (std/mean)
    prize_mean = np.mean(prize)
    prize_std = np.std(prize)
    prize_cv = prize_std / (prize_mean + 1e-9)  # Avoid division by zero
    
    # Maximum distance from depot (graph radius)
    max_dist_from_depot = np.max(distance[1:, 0]) if n > 1 else 0  # Exclude depot from max calculation
    
    # Distance-prize correlation (between prize[j] and distance[0,j])
    depot_distances = distance[1:, 0] if n > 1 else np.array([0])
    node_prizes = prize[1:] if n > 1 else prize
    if len(depot_distances) > 1:
        # Calculate correlation manually to avoid importing scipy
        mean_dist = np.mean(depot_distances)
        mean_prize = np.mean(node_prizes)
        numerator = np.sum((depot_distances - mean_dist) * (node_prizes - mean_prize))
        denominator = np.sqrt(np.sum((depot_distances - mean_dist)**2) * np.sum((node_prizes - mean_prize)**2))
        dist_prize_corr = numerator / (denominator + 1e-9)
    else:
        dist_prize_corr = 0
    
    # Adaptive neighborhood size based on instance features
    # Base neighborhood size influenced by prize CV and distance-prize correlation
    base_neighborhood_size = 5
    # Increase neighborhood if prizes are highly variable or negatively correlated with distance
    cv_factor = min(max(1.0, prize_cv * 0.5), 2.0)  # Cap between 0.5 and 2.0
    corr_factor = min(max(1.0, (1 - dist_prize_corr) * 0.5), 2.0)  # If distant nodes have higher prizes
    adaptive_k = int(np.clip(base_neighborhood_size * cv_factor * corr_factor, 3, 7))
    
    # Initialize the heuristic matrix
    heuristics_matrix = np.zeros((n, n))
    
    # Calculate the basic heuristic value (prize/distance ratio) for each possible move
    # Avoid division by zero by ensuring distance is not zero
    basic_heuristic = np.zeros((n, n))
    non_zero_distances = distance.copy()
    # Set diagonal to a large value to avoid self loops
    np.fill_diagonal(non_zero_distances, np.inf)
    
    for i in range(n):
        for j in range(n):
            if i != j:  # Don't consider moving from a node to itself
                basic_heuristic[i, j] = prize[j] / non_zero_distances[i, j]
    
    # Calculate neighborhood prize density for each node with adaptive neighborhood size
    # This will be used later in the heuristic calculation
    neighbor_densities = np.zeros(n)
    for j in range(n):
        all_other_nodes = np.arange(n)
        other_nodes_except_j = all_other_nodes[all_other_nodes != j]
        sorted_by_distance_to_j = other_nodes_except_j[np.argsort(distance[j, other_nodes_except_j])]
        j_neighbors = sorted_by_distance_to_j[:min(adaptive_k, len(sorted_by_distance_to_j))]
        
        avg_neighbor_prize = np.mean(prize[j_neighbors]) if len(j_neighbors) > 0 else prize[j]
        neighbor_densities[j] = avg_neighbor_prize

    # Apply sparsification: for each node i, keep only the top-k most promising edges
    for i in range(n):
        # Get the heuristic values for node i to all other nodes
        node_heuristics = basic_heuristic[i, :].copy()
        
        # Temporarily set the diagonal element to a very small value to exclude self-loop
        node_heuristics[i] = -np.inf
        
        # Find the indices of the top-k most promising nodes
        k = min(10, n - 1) if n > 1 else n - 1
        if k >= n - 1:
            top_k_indices = np.arange(n)
            top_k_indices = top_k_indices[top_k_indices != i]
        else:
            top_k_indices = np.argpartition(node_heuristics, -k)[-k:]
        
        # For each of the top-k indices, calculate a refined heuristic
        for j in top_k_indices:
            if j != i:  # Double-check to exclude self-loop
                # Basic heuristic value
                basic_value = node_heuristics[j]
                
                # Calculate squared prize/distance ratio to emphasize high-efficiency moves
                squared_efficiency = (prize[j] / (distance[i, j] + 1e-9)) ** 2
                
                # Use the pre-calculated neighborhood density
                neighborhood_factor = 1 + neighbor_densities[j] / (prize_mean + 1e-9)
                
                # The resulting heuristic combines multiple exploration-friendly signals:
                # - Basic value: prize[j]/distance[i,j]
                # - Squared efficiency: emphasizes high-efficiency moves
                # - Neighborhood value: prize density around node j with adaptive neighborhood size
                heuristics_matrix[i, j] = basic_value * squared_efficiency * neighborhood_factor
    
    # Ensure no negative values and handle any remaining zeros
    heuristics_matrix = np.maximum(heuristics_matrix, 0)
    
    # Small epsilon to avoid zero values which could cause issues in the ACO algorithm
    # Only set to epsilon where there's still a zero after our calculations
    zero_mask = (heuristics_matrix == 0)
    heuristics_matrix[zero_mask] = 1e-9
    
    return heuristics_matrix

## Driving problem
Solution found at round 12 (ID: 61).

In [14]:
import math
from typing import Dict, List, Tuple
from collections import defaultdict


def driving_actions(edge_info: dict, vehicles_info: list[dict]) -> list[dict]:
    """
    Density-adaptive constant time-gap following algorithm with dynamic parameter adjustment.
    
    Implementation Idea:
    - Calculate real-time lane-specific traffic density metrics each timestep
    - Categorize density into three levels: high (>0.3 veh/m), medium (0.15-0.3 veh/m), low (<0.15 veh/m)
    - Adjust car-following parameters dynamically based on density:
        * High density: Conservative baseline (time_headway=3.0s, min_gap=6.0m, target_speed=0.35 m/s)
        * Medium density: Moderate (time_headway=2.5s, min_gap=5.0m, target_speed=1.5 m/s) 
        * Low density: Aggressive (time_headway=2.0s, min_gap=4.0m, target_speed=min(5.0, max_speed))
    - Smooth parameter transitions to prevent abrupt behavior changes
    - Maintain proven constant time-gap framework with deadlock prevention
    - No lane changes to maintain stability
    
    Implementation Considerations:
    1. Density calculation uses lane-specific vehicle counts divided by edge length
    2. Parameter smoothing uses exponential moving average to prevent abrupt changes
    3. Critical TTC monitoring ensures safety margins are maintained
    4. Deadlock prevention accelerates vehicles stationary >10 steps with sufficient gap
    5. Action smoothing limits acceleration changes to 0.5 m/s² per step
    
    Args:
        edge_info (dict): Target edge information
        vehicles_info (list[dict]): Information about vehicles on the edge
        
    Returns:
        list[dict]: Actions for each vehicle
    """
    
    # === PERSISTENT STATE INITIALIZATION ===
    if not hasattr(driving_actions, 'state'):
        driving_actions.state = {
            'step_counter': 0,
            'vehicle_stationary_steps': defaultdict(int),
            'last_actions': {}
        }
    
    state = driving_actions.state
    state['step_counter'] += 1
    
    # === ALGORITHM PARAMETERS ===
    # Fixed conservative parameters (parent solution)
    TIME_HEADWAY = 3.0    # seconds
    MIN_GAP = 6.0         # meters
    TARGET_SPEED = 0.35   # m/s
    
    # Deadlock prevention parameters
    STATIONARY_THRESHOLD = 5  # steps before deadlock intervention
    DEADLOCK_ACCEL = 1.0  # m/s² acceleration for deadlocked vehicles
    
    # Vehicle characteristics
    VEHICLE_CHAR = {
        'passenger': {'length': 5.0, 'max_accel': 2.6, 'max_decel': 4.5},
        'bus': {'length': 12.0, 'max_accel': 1.2, 'max_decel': 4.0},
        'truck': {'length': 7.5, 'max_accel': 1.3, 'max_decel': 4.0},
        'emergency': {'length': 5.0, 'max_accel': 2.6, 'max_decel': 4.5}
    }
    
    # === HELPER FUNCTIONS ===
    
    def get_vehicle_char(veh_type: str) -> dict:
        """Get vehicle characteristics"""
        return VEHICLE_CHAR.get(veh_type, VEHICLE_CHAR['passenger'])
    
    def get_vehicle_length(veh_type: str) -> float:
        """Get vehicle length"""
        return get_vehicle_char(veh_type)['length']
    
    def get_leader(vehicle: dict, all_vehicles: list[dict]) -> dict:
        """Find the immediate leader in the same lane"""
        leader = None
        min_distance = float('inf')
        
        current_lane = vehicle['current_lane']
        current_pos = vehicle['position']
        
        for other in all_vehicles:
            if (other['current_lane'] == current_lane and 
                other['position'] > current_pos):
                distance = other['position'] - current_pos
                if distance < min_distance:
                    min_distance = distance
                    leader = other
        
        return leader
    
    def calculate_lane_density(lane_id: str, vehicles: list[dict]) -> float:
        """Calculate current density for a specific lane (vehicles per meter)"""
        # Count vehicles in this lane
        lane_vehicles = [v for v in vehicles if v['current_lane'] == lane_id]
        
        if not lane_vehicles:
            return 0.0
        
        # Edge length is used as approximation for lane length
        edge_length = edge_info.get('length', 100.0)
        
        # Return density (vehicles per meter)
        return len(lane_vehicles) / edge_length
    
    def get_density_category(density: float) -> str:
        """Categorize density into high, medium, or low"""
        if density > HIGH_DENSITY_THRESHOLD:
            return 'high'
        elif density >= MEDIUM_DENSITY_THRESHOLD:
            return 'medium'
        else:
            return 'low'
    
    def smooth_density(lane_id: str, current_density: float) -> float:
        """Apply exponential moving average to smooth density measurements"""
        if lane_id not in state['lane_density_ema']:
            state['lane_density_ema'][lane_id] = current_density
        else:
            state['lane_density_ema'][lane_id] = (
                DENSITY_SMOOTHING_ALPHA * current_density + 
                (1 - DENSITY_SMOOTHING_ALPHA) * state['lane_density_ema'][lane_id]
            )
        return state['lane_density_ema'][lane_id]
    
    def get_vehicle_parameters(vehicle: dict, lane_density: float) -> dict:
        """Get car-following parameters for a vehicle based on lane density"""
        # Get density category
        density_cat = get_density_category(lane_density)
        
        # Get base parameters for this density category
        base_params = PARAMS_BY_DENSITY[density_cat].copy()
        
        # Smooth parameter transitions if vehicle has previous parameters
        veh_id = vehicle['veh_id']
        if veh_id in state['vehicle_params']:
            prev_params = state['vehicle_params'][veh_id]
            
            # Smooth time_headway transition (max change 0.1s per step)
            time_headway_diff = base_params['time_headway'] - prev_params['time_headway']
            time_headway_change = max(-0.1, min(0.1, time_headway_diff))
            base_params['time_headway'] = prev_params['time_headway'] + time_headway_change
            
            # Smooth min_gap transition (max change 0.1m per step)
            min_gap_diff = base_params['min_gap'] - prev_params['min_gap']
            min_gap_change = max(-0.1, min(0.1, min_gap_diff))
            base_params['min_gap'] = prev_params['min_gap'] + min_gap_change
            
            # Smooth target_speed transition (max change 0.1 m/s per step)
            target_speed_diff = base_params['target_speed'] - prev_params['target_speed']
            target_speed_change = max(-0.1, min(0.1, target_speed_diff))
            base_params['target_speed'] = prev_params['target_speed'] + target_speed_change
        
        # Store for next step
        state['vehicle_params'][veh_id] = base_params.copy()
        
        return base_params
    
    def calculate_target_speed(vehicle: dict, leader: dict, params: dict) -> float:
        """
        Calculate target speed using constant time-gap following with given parameters
        
        Based on: desired_gap = MIN_GAP + vehicle['speed'] * TIME_HEADWAY
        """
        if leader is None:
            return params['target_speed']
        
        # Calculate current gap to leader
        current_gap = leader['position'] - vehicle['position'] - get_vehicle_length(leader['veh_type'])
        
        # Calculate desired gap based on constant time headway
        desired_gap = params['min_gap'] + vehicle['speed'] * params['time_headway']
        
        # If we have sufficient gap, aim for target speed
        if current_gap >= desired_gap * 1.1:  # 10% buffer
            # Cap at leader's speed plus small margin
            return min(params['target_speed'], leader['speed'] * 1.1)
        
        # If gap is too small, adjust speed to maintain safe gap
        elif current_gap < desired_gap:
            # Calculate speed adjustment needed
            gap_error = desired_gap - current_gap
            # Convert gap error to speed reduction (divide by time headway)
            speed_adjustment = gap_error / params['time_headway']
            target_speed = max(0, leader['speed'] - speed_adjustment)
            return min(target_speed, params['target_speed'])
        
        # Maintain current speed if gap is acceptable
        else:
            return min(vehicle['speed'], params['target_speed'])
    
    def calculate_acceleration(vehicle: dict, target_speed: float) -> float:
        """Calculate acceleration to reach target speed with smoothing"""
        current_speed = vehicle['speed']
        veh_char = get_vehicle_char(vehicle['veh_type'])
        
        # Simple proportional control
        speed_error = target_speed - current_speed
        
        # Use 1-second time constant for responsive control
        base_accel = speed_error  # m/s²
        
        # Apply vehicle-specific limits
        base_accel = max(-veh_char['max_decel'], min(base_accel, veh_char['max_accel']))
        
        # Smooth acceleration changes using last action
        if vehicle['veh_id'] in state['last_actions']:
            last_accel = state['last_actions'][vehicle['veh_id']].get('acceleration', 0)
            # Limit acceleration change to 0.5 m/s² per step for smoothness
            max_change = 0.5
            base_accel = max(last_accel - max_change, min(last_accel + max_change, base_accel))
        
        return base_accel
    
    # === MAIN ALGORITHM EXECUTION ===
    
    # No density adaptation - use fixed parameters
    # (keeping lane_ids for potential future use)
    lane_ids = [f"{edge_info['edge_id']}_{i}" for i in range(edge_info['lane_count'])]
    
    # Update stationary step counter for deadlock detection
    current_veh_ids = {v['veh_id'] for v in vehicles_info}
    
    # Clean up departed vehicles
    for veh_id in list(state['vehicle_stationary_steps'].keys()):
        if veh_id not in current_veh_ids:
            state['vehicle_stationary_steps'].pop(veh_id, None)
    
    # Track stationary vehicles
    for veh in vehicles_info:
        veh_id = veh['veh_id']
        if veh['speed'] < 0.1:  # Nearly stationary
            state['vehicle_stationary_steps'][veh_id] += 1
        else:
            state['vehicle_stationary_steps'][veh_id] = 0
    
    # Determine actions for each vehicle
    actions = []
    
    for veh in vehicles_info:
        veh_id = veh['veh_id']
        
        # Use fixed parameters (no density adaptation)
        params = {
            'time_headway': TIME_HEADWAY,
            'min_gap': MIN_GAP,
            'target_speed': TARGET_SPEED
        }
        
        # Find leader in same lane
        leader = get_leader(veh, vehicles_info)
        
        # Calculate target speed using adaptive parameters
        target_speed = calculate_target_speed(veh, leader, params)
        
        # Deadlock prevention: if vehicle has been stationary too long
        if state['vehicle_stationary_steps'].get(veh_id, 0) > STATIONARY_THRESHOLD:
            # Check if there's enough space ahead (at least min_gap)
            if leader is None or (leader['position'] - veh['position'] > params['min_gap']):
                # Force acceleration to break deadlock
                target_speed = max(target_speed, min(params['target_speed'], 0.5))
                # Reset stationary counter
                state['vehicle_stationary_steps'][veh_id] = 0
        
        # Emergency vehicles: no special boost for safety (parent solution had none)
        # (We keep the same target_speed)
        
        # Calculate acceleration
        acceleration = calculate_acceleration(veh, target_speed)
        
        # No lane changes to maintain stability (proven effective in parent solutions)
        lane_changing = 0
        
        # Store action for smoothing in next step
        state['last_actions'][veh_id] = {
            'acceleration': acceleration,
            'lane_changing': lane_changing
        }
        
        actions.append({
            'veh_id': veh_id,
            'acceleration': acceleration,
            'lane_changing': lane_changing
        })
    
    return actions

Final optimal solution:

In [2]:
import traci

def driving_actions(edge_info, vehicles_info) -> list[dict]:
    """
    Decide the next actions for all vehicles for the current time step.
    
    Args:
        edge_info (dict): target edge info; dict keys are:
        - 'edge_id' (str)
        - 'lane_count' (int)
        - 'length' (float, m)
        - 'max_speed' (float, m/s)

        vehicles_info (list[dict]): info of vehicles on the edge; one dict for each vehicle; dict keys are:
        - 'veh_id' (str)
        - 'veh_type' (str)
        - 'position' (float) - x-coordinate position
        - 'speed' (float, m/s)
        - 'current_lane' (str) - e.g., "E0_0"
        - 'wants_left'/'wants_right' (bool)
        - 'potential_target_lanes' (list[tuple]) - (laneID, length, occupation, bestLaneOffset, allowsContinuation, nextLanes)

    Returns:
        actions (list[dict]): actions to take; one dict for each vehicle; dict keys are:
        - 'veh_id'
        - 'acceleration' (float, m/s^2); positive for acceleration, negative for deceleration
        - 'lane_changing' (int: +1=left, 0=keep, -1=right)
    """
    # =====PARAMETERS & VARIABLES=====
    # -----CONSTANTS-----
    # Safety parameters
    SAFE_TTC = 3.0                      # Safe time-to-collision threshold (seconds)
    MIN_GAP = 1.6                       # Minimum gap for car following (meters); Note: gap does not include vehicle length
    LANE_CHANGE_GAP_BASE = 10.0         # Base gap required for lane changes (meters); Note: gap does not include vehicle length
    OFFSET_BUFFER = 1.6                 # Safety buffer added to vehicle length for lane-changing (meters); Note: buffer does not include vehicle length
    
    # Entry behavior parameters
    ENTRY_DISTANCE = 15.0               # Distance from start considered as entry zone (meters)
    ENTRY_TIME = 3.0                    # Time after entering considered as entry period (seconds)
    
    # Lane change control parameters
    LANE_CHANGE_COOLDOWN = 2.5          # Minimum time between lane changes (seconds)
    TARGET_LANE_PATIENCE = 60.0         # Distance to stay in target lane before changing again (meters)
    
    # Urgency calculation parameters
    URGENCY_START_DIST = 400.0          # Distance from end where urgency starts increasing (meters)
    CRITICAL_DIST = 80.0                # Distance from end where urgency reaches maximum (meters)
    
    # Acceleration parameters
    MIN_ACCEL = 0.3                     # Minimum acceleration to maintain (m/s²)
    MAX_ACCEL_BOOST = 1.2               # Multiplier for maximum acceleration in free flow
    
    # Cooperation parameters
    COOPERATION_RANGE = 80.0           # Range for cooperative behavior (meters)
    COOPERATION_WINDOW = 6.0            # Time window for cooperative intent signaling (seconds)
    SPEED_MATCH_FACTOR = 0.6            # Factor for speed matching adjustments
    
    # Gap creation parameters
    GAP_SEARCH_ACCEL = 3.2              # Acceleration when searching for gaps (m/s²)
    GAP_SEARCH_DECEL = 1.8              # Deceleration when searching for gaps (m/s²)
    
    # Swap coordination parameters
    SWAP_DETECT_RANGE = 50.0            # Range to detect swap conflicts (meters)
    SWAP_COORDINATION_ACCEL = 2.5       # Acceleration for swap coordination (m/s²)
    SWAP_COORDINATION_DECEL = 2.0       # Deceleration for swap coordination (m/s²)
    
    # Platoon formation parameters
    PLATOON_DETECT_RANGE = 18.0         # Range to detect platoon opportunities (meters)
    PLATOON_ACCEL = 3.0                 # Acceleration adjustment for platoon coordination (m/s²)
    PLATOON_DECEL = 1.5                 # Deceleration adjustment for platoon coordination (m/s²)
    
    # -----VEHICLE TYPE PARAMETERS-----
    
    # Vehicle-specific lengths (meters)
    veh_length = {
        "passenger": 5.0,
        "bus": 12.0,
        "truck": 7.1,
        "emergency": 6.5
    }
    
    # Vehicle-specific acceleration capabilities (m/s²)
    max_accel = {
        "passenger": 2.6,
        "bus": 1.2,
        "truck": 1.3,
        "emergency": 2.6
    }
    
    # Vehicle-specific deceleration capabilities (m/s²)
    max_decel = {
        "passenger": 4.5,
        "bus": 4.0,
        "truck": 4.0,
        "emergency": 4.5
    }
    
    # -----MEMORY INITIALIZATION-----
    
    if not hasattr(driving_actions, 'last_lane_change'):
        driving_actions.last_lane_change = {}
    
    if not hasattr(driving_actions, 'entry_time'):
        driving_actions.entry_time = {}
    
    if not hasattr(driving_actions, 'target_lane_reached'):
        driving_actions.target_lane_reached = {}
    
    if not hasattr(driving_actions, 'lane_change_intent'):
        driving_actions.lane_change_intent = {}
    
    if not hasattr(driving_actions, 'swap_partners'):
        driving_actions.swap_partners = {}
    
    if not hasattr(driving_actions, 'platoon_pairs'):
        driving_actions.platoon_pairs = {}
    
    current_time = traci.simulation.getTime()
    
    #-----ORGANIZE VEHICLES BY LANE-----
    
    lanes = {}
    for i in range(edge_info['lane_count']):
        lanes[f"{edge_info['edge_id']}_{i}"] = []
    
    for veh in vehicles_info:
        lanes[veh['current_lane']].append(veh)
    
    for lane in lanes:
        lanes[lane].sort(key=lambda x: x['position'], reverse=True)  
        # Note: vehicles are sorted by position, largest first
        # this makes it easy to iterate vehicle from upstream to downstream
    
    # =====HELPER FUNCTIONS=====
    
    def compute_accel(veh_speed, veh_type, leader_speed, gap, max_speed):
        """Compute optimal acceleration based on IDM"""
        if leader_speed is None or gap is None:
            return min(max_accel[veh_type] * MAX_ACCEL_BOOST, max_speed - veh_speed)
        
        if gap < MIN_GAP:
            return -max_decel[veh_type]
        
        desired_gap = max(MIN_GAP, 1.5 * veh_speed + 2.0)  # desired gap calculation; Note: gap does not include vehicle length
        relative_speed = veh_speed - leader_speed
        
        if relative_speed > 0:
            ttc = gap / relative_speed
            if ttc < SAFE_TTC:
                decel = -min(max_decel[veh_type], relative_speed / SAFE_TTC * 1.2)
                return max(-max_decel[veh_type], decel)
        
        if gap < desired_gap:
            return max(-max_decel[veh_type] * 0.4, -0.6 * (desired_gap - gap) / SAFE_TTC)
        
        return min(max_accel[veh_type] * MAX_ACCEL_BOOST, max_speed - veh_speed)
    
    def get_target_lane(veh):
        """Determine target lane based on route"""
        lane_index = int(veh['current_lane'].split('_')[-1])
        
        for lane_info in veh['potential_target_lanes']:
            if lane_info[3] == 0 and lane_info[4]:
                return int(lane_info[0].split('_')[-1])
        
        return lane_index
    
    def get_current_lane_length(veh):
        """Determine target lane based on route"""
        ans = edge_info['length']  # default to full length if not found
        
        for lane_info in veh['potential_target_lanes']:
            if lane_info[0] == veh['current_lane']:
                return float(lane_info[1])
        return ans
    
    def is_lane_change_safe(veh, target_lane_id, lanes, urgency, min_gap_override=None):
        """Check if lane change is safe with relaxed requirements for safe speeds"""
        position = veh['position']
        speed = veh['speed']
        veh_type = veh['veh_type']
        my_length = veh_length[veh_type]
        
        if min_gap_override is not None:
            gap_threshold = min_gap_override
        else:
            gap_threshold = LANE_CHANGE_GAP_BASE * (1.0 - 0.35 * urgency)
        
        ttc_threshold = SAFE_TTC * (1.0 - 0.25 * urgency)
        
        target_vehs = lanes.get(target_lane_id, [])
        
        # Check leader in target lane
        vehicles_ahead = [v for v in target_vehs if v['position'] > position]
        vehicles_ahead = sorted(vehicles_ahead, key=lambda x: x['position'])
        leader = vehicles_ahead[0] if vehicles_ahead else None
        if leader:
            leader_length = veh_length[leader['veh_type']]
            gap = leader['position'] - position - leader_length
            rel_speed = speed - leader['speed']
            
            # If we're slower or equal speed, just need one vehicle length
            if rel_speed <= 0 and gap >= OFFSET_BUFFER:
                pass  # Safe
            else:
                if gap < gap_threshold:
                    return False
                if rel_speed > 0:
                    ttc = gap / rel_speed
                    if ttc < ttc_threshold:
                        return False
        
        # Check follower in target lane
        follower = next((v for v in target_vehs if v['position'] < position), None)
        if follower:
            gap = position - follower['position'] - my_length
            rel_speed = follower['speed'] - speed
            
            # If follower is slower or equal speed, just need one vehicle length
            if rel_speed <= 0 and gap >= OFFSET_BUFFER:
                pass  # Safe
            else:
                if gap < gap_threshold * 0.75:
                    return False
                if rel_speed > 0:
                    ttc = gap / rel_speed
                    if ttc < ttc_threshold:
                        return False
        
        return True
    
    def detect_swap_conflict(veh1, veh2, target1, target2):
        """Detect if two vehicles want to swap lanes"""
        lane1 = int(veh1['current_lane'].split('_')[-1])
        lane2 = int(veh2['current_lane'].split('_')[-1])
        
        # Target lanes may be more than one lane away; but vehicle can only change one lane at a time
        # so we normalize to adjacent lanes
        if target1 > lane1:
            target1 = lane1 + 1
        elif target1 < lane1:
            target1 = lane1 - 1
        if target2 > lane2:
            target2 = lane2 + 1
        elif target2 < lane2:
            target2 = lane2 - 1
            
        if abs(lane1 - lane2) == 1:
            if target1 == lane2 and target2 == lane1:
                pos_diff = abs(veh1['position'] - veh2['position'])
                if pos_diff < SWAP_DETECT_RANGE:
                    return True
        
        return False
    
    def detect_platoon_opportunity(veh1, veh2, target1, target2):
        """
        Detect if two vehicles can form a platoon (FLEXIBLE VERSION)
        
        This version supports bidirectional platoon formation:
        - If follower is behind: lane vehicle accelerates, joining vehicle decelerates
        - If follower is ahead: lane vehicle decelerates, joining vehicle accelerates
        
        Returns:
            (lane_veh_id, joining_veh_id, position_relation) tuple if platoon possible
            position_relation: 'lane_ahead' if lane_vehicle is ahead, 'lane_behind' if behind
            None if no platoon opportunity
        """
        lane1 = int(veh1['current_lane'].split('_')[-1])
        lane2 = int(veh2['current_lane'].split('_')[-1])
            
        if current_time == 215 and veh1['veh_id'] in ['veh_1_185_0', 'veh_4_192_0'] and veh2['veh_id'] in ['veh_1_185_0', 'veh_4_192_0']:  # for debugging
            print('', end='')  # put a breakpoint here if needed
        
        # Target lanes may be more than one lane away; but vehicle can only change one lane at a time
        # so we normalize to adjacent lanes
        if target1 > lane1:
            target1 = lane1 + 1
        elif target1 < lane1:
            target1 = lane1 - 1
        if target2 > lane2:
            target2 = lane2 + 1
        elif target2 < lane2:
            target2 = lane2 - 1
            
        # Must have same target lane
        if target1 != target2:
            return None
        
        # Must be in adjacent lanes
        if abs(lane1 - lane2) != 1:
            return None
        
        # Check if close enough
        pos_diff = abs(veh1['position'] - veh2['position'])
        if pos_diff > PLATOON_DETECT_RANGE:
            return None
        
        threshold = 0.8  # threshold to decide that vehicles are very close (meters); then we need their types to decide who should lead
        # Determine who is in target lane and who needs to join
        if lane1 == target1 and lane2 != target2:
            # veh1 is in target lane, veh2 needs to join
            lane_veh_id = veh1['veh_id']
            joining_veh_id = veh2['veh_id']
            if veh1['position'] - veh2['position'] >= threshold:
                position_relation = 'lane_ahead'
            elif (0 <= veh1['position'] - veh2['position'] < threshold) and not (veh1['veh_type'] in ['bus', 'truck'] and veh2['veh_type'] in ['passenger', 'emergency']):
                position_relation = 'lane_ahead'
            elif -threshold <= veh1['position'] - veh2['position'] < 0 and (veh1['veh_type'] in ['passenger', 'emergency']and veh2['veh_type'] in ['bus', 'truck']):
                position_relation = 'lane_ahead'
            else:
                position_relation = 'lane_behind'
            return (lane_veh_id, joining_veh_id, position_relation)
            
        elif lane2 == target2 and lane1 != target1:
            # veh2 is in target lane, veh1 needs to join
            lane_veh_id = veh2['veh_id']
            joining_veh_id = veh1['veh_id']
            if veh2['position'] - veh1['position'] >= threshold:
                position_relation = 'lane_ahead'
            elif (0 <= veh2['position'] - veh1['position'] < threshold) and not (veh2['veh_type'] in ['bus', 'truck'] and veh1['veh_type'] in ['passenger', 'emergency']):
                position_relation = 'lane_ahead'
            elif -threshold <= veh2['position'] - veh1['position'] < 0 and (veh2['veh_type'] in ['passenger', 'emergency'] and veh1['veh_type'] in ['bus', 'truck']):
                position_relation = 'lane_ahead'
            else:
                position_relation = 'lane_behind'
            return (lane_veh_id, joining_veh_id, position_relation)
        
        return None
    
    def update_platoon(platoon_pairs, lane_veh_id, joining_veh_id, position_relation):
        """ 
        Avoid duplicate platoon entries
        Check if this pair is already recorded in either order
        if so, decide whether current platoon pair is better
        """
        joining_veh = vehicle_states[joining_veh_id]['veh']
        joining_veh_position = joining_veh['position']
        
        lane_veh = vehicle_states[lane_veh_id]['veh']
        lane_veh_position = lane_veh['position']
        
        if current_time == 215 and joining_veh['veh_id'] in ['veh_1_185_0'] and lane_veh['veh_id'] in ['veh_4_192_0', 'veh_1_186_0']:  # for debugging
            print('', end='')  # put a breakpoint here if needed
        
        # absolute difference between the new platoon pair proposal
        abs_dist = abs(lane_veh_position - joining_veh_position)
        
        # The following logic ensures that a joining vehicle only forms a platoon with the closest lane vehicle
        # we need to check `driving_actions.platoon_pairs`
        for i, pair in enumerate(platoon_pairs):
            if joining_veh_id == pair[1]:
                last_lane_veh_id = pair[0]
                last_lane_veh = vehicle_states[last_lane_veh_id]['veh']
                last_lane_veh_position = last_lane_veh['position']
                
                if abs_dist < abs(last_lane_veh_position - joining_veh_position):
                    # if new platoon pair is closer, replace the old one
                    # remove old pair from platoon_pairs
                    platoon_pairs.pop(i)
                    platoon_pairs.append((lane_veh_id, joining_veh_id, position_relation,
                                        vehicle_states[lane_veh_id], vehicle_states[joining_veh_id]))
                    return platoon_pairs
                else:
                    return platoon_pairs # keep the old one
        # if no conflict, just add the new pair
        platoon_pairs.append((lane_veh_id, joining_veh_id, position_relation,
                             vehicle_states[lane_veh_id], vehicle_states[joining_veh_id]))
        return platoon_pairs
                
    
    def calculate_min_offset(leader_veh_type):
        """
        Calculate minimum offset required for safe lane change
        Uses the length of the leader vehicle plus a safety buffer.
            
        Returns:
            Minimum offset distance in meters
        """
        # Use the larger vehicle length plus safety buffer
        return veh_length[leader_veh_type] + OFFSET_BUFFER
    
    # =====MEMORY CLEANUP=====
    
    actions = []
    current_veh_ids = set(veh['veh_id'] for veh in vehicles_info)
    
    driving_actions.last_lane_change = {k: v for k, v in driving_actions.last_lane_change.items() if k in current_veh_ids}
    driving_actions.entry_time = {k: v for k, v in driving_actions.entry_time.items() if k in current_veh_ids}
    driving_actions.target_lane_reached = {k: v for k, v in driving_actions.target_lane_reached.items() if k in current_veh_ids}
    driving_actions.lane_change_intent = {k: v for k, v in driving_actions.lane_change_intent.items() if k in current_veh_ids and current_time - v['time'] < COOPERATION_WINDOW}
    driving_actions.swap_partners = {k: v for k, v in driving_actions.swap_partners.items() if k in current_veh_ids and v in current_veh_ids}
    driving_actions.platoon_pairs = {k: v for k, v in driving_actions.platoon_pairs.items() if k in current_veh_ids and v in current_veh_ids}
    
    # =====FIRST PASS: ANALYZE SITUATION=====
    
    vehicle_states = {}
    swap_pairs = []
    platoon_pairs = []
    
    for veh in vehicles_info:
        veh_id = veh['veh_id']
        
        if traci.simulation.getTime() == 66 and veh_id in ['veh_5_52_0']:  # for debugging
            print('', end='')  # put a breakpoint here if needed
            
        position = veh['position']
        lane_index = int(veh['current_lane'].split('_')[-1])
        
        if veh_id not in driving_actions.entry_time:
            driving_actions.entry_time[veh_id] = current_time
        
        entry_age = current_time - driving_actions.entry_time[veh_id]
        is_entry = position < ENTRY_DISTANCE or entry_age < ENTRY_TIME
        
        target_lane_index = get_target_lane(veh)
        dist_to_end = get_current_lane_length(veh) - position
        
        if dist_to_end > URGENCY_START_DIST:
            urgency = 0.0
        elif dist_to_end > CRITICAL_DIST:
            urgency = (URGENCY_START_DIST - dist_to_end) / (URGENCY_START_DIST - CRITICAL_DIST)
        else:
            urgency = 1.0
        
        vehicle_states[veh_id] = {
            'veh': veh,
            'is_entry': is_entry,
            'target_lane_index': target_lane_index,
            'urgency': urgency,
            'needs_change': lane_index != target_lane_index and not is_entry,
            'swap_role': None,
            'swap_partner': None,
            'platoon_role': None,
            'platoon_partner': None,
            'platoon_relation': None
        }
    
    # -----DETECT SWAP CONFLICTS-----
    
    veh_list = list(vehicle_states.keys())
    for i in range(len(veh_list)):
        for j in range(i + 1, len(veh_list)):
            veh1_id = veh_list[i]
            veh2_id = veh_list[j]
            
            if traci.simulation.getTime() == 417 and (veh1_id in ['veh_3_404_0', 'veh_1_397_0'] and veh2_id in ['veh_3_404_0', 'veh_1_397_0']):  # for debugging
                print('', end='')  # put a breakpoint here if needed
            
            state1 = vehicle_states[veh1_id]
            state2 = vehicle_states[veh2_id]
            
            if state1['needs_change'] and state2['needs_change']:
                if detect_swap_conflict(state1['veh'], state2['veh'], 
                                       state1['target_lane_index'], state2['target_lane_index']):
                    swap_pairs.append((veh1_id, veh2_id, state1, state2))
    
    # -----DETECT PLATOON OPPORTUNITIES-----

    for i in range(len(veh_list)):
        for j in range(i + 1, len(veh_list)):
            veh1_id = veh_list[i]
            veh2_id = veh_list[j]
            
            if traci.simulation.getTime() == 20 and veh1_id in ['veh_6_9_0', 'veh_4_5_0'] and veh2_id in ['veh_6_9_0', 'veh_4_5_0']:  # for debugging
                print('', end='')  # put a breakpoint here if needed
                
            state1 = vehicle_states[veh1_id]
            state2 = vehicle_states[veh2_id]
            
            # Skip if already in swap coordination
            if state1.get('swap_partner') or state2.get('swap_partner'):
                continue
            
            # Check for platoon opportunity
            platoon_result = detect_platoon_opportunity(
                state1['veh'], state2['veh'],
                state1['target_lane_index'], state2['target_lane_index']
            )
            
            if platoon_result:
                lane_veh_id, joining_veh_id, position_relation = platoon_result
                platoon_pairs = update_platoon(platoon_pairs, lane_veh_id, joining_veh_id, position_relation)
                # Avoid duplicate platoon entries
                # Check if this pair is already recorded in either order
                # if so, decide whether current platoon pair is better
    
    
    # -----CHECK SWAP & PLATOON DUPLICATION-----
    # If a vehicle is in swap_pairs, namely this vehicle want to swap lane with another vehicle,
    # at the same time, this vehicle is in platoon_pairs, namely this vehicle want to change lane to join another vehicle to form a platoon,
    # this creates a conflict, we need to resolve it by removing one action
    # this is done by compare the position of the vehicle to swap and the position of the "lane vehicle" in the platoon pair
    # the closer one gets to keep its action, the other one is removed
    
    # Build maps for easier lookup
    swap_map = {}  # {veh_id: (partner_id, state1, state2)}
    for veh1_id, veh2_id, state1, state2 in swap_pairs:
        swap_map[veh1_id] = (veh2_id, state1, state2)
        swap_map[veh2_id] = (veh1_id, state2, state1)
    
    platoon_map = {}  # {veh_id: (partner_id, relation, my_state, partner_state)}
    for lane_veh_id, joining_veh_id, position_relation, lane_state, joining_state in platoon_pairs:
        #platoon_map[lane_veh_id] = (joining_veh_id, position_relation, lane_state, joining_state)
        platoon_map[joining_veh_id] = (lane_veh_id, position_relation, joining_state, lane_state)
    
    # Find vehicles that are in both swap and platoon pairs
    conflicting_vehicles = set(swap_map.keys()) & set(platoon_map.keys())
    
    vehicles_to_remove_from_swap = set()
    vehicles_to_remove_from_platoon = set()
    
    for veh_id in conflicting_vehicles:
        # Get swap partner info
        swap_partner_id, my_swap_state, partner_swap_state = swap_map[veh_id]
        swap_partner_veh = my_swap_state['veh'] if my_swap_state['veh']['veh_id'] == veh_id else partner_swap_state['veh']
        actual_swap_partner_veh = partner_swap_state['veh'] if my_swap_state['veh']['veh_id'] == veh_id else my_swap_state['veh']
        
        # Get platoon partner info
        platoon_partner_id, platoon_relation, my_platoon_state, partner_platoon_state = platoon_map[veh_id]
        platoon_partner_veh = my_platoon_state['veh'] if my_platoon_state['veh']['veh_id'] == veh_id else partner_platoon_state['veh']
        actual_platoon_partner_veh = partner_platoon_state['veh'] if my_platoon_state['veh']['veh_id'] == veh_id else my_platoon_state['veh']
        
        # Get current vehicle info
        current_veh = my_swap_state['veh'] if my_swap_state['veh']['veh_id'] == veh_id else partner_swap_state['veh']
        
        '''
        current_position = current_veh['position']
        # Calculate distances to both partners
        swap_distance = abs(current_position - actual_swap_partner_veh['position'])
        platoon_distance = abs(current_position - actual_platoon_partner_veh['position'])
        
        # Keep the action with the closer partner
        if swap_distance < platoon_distance:
            # Keep swap, remove platoon
            vehicles_to_remove_from_platoon.add(veh_id)
            if platoon_partner_id in conflicting_vehicles:
                vehicles_to_remove_from_platoon.add(platoon_partner_id)
        else:
            # Keep platoon, remove swap
            vehicles_to_remove_from_swap.add(veh_id)
            if swap_partner_id in conflicting_vehicles:
                vehicles_to_remove_from_swap.add(swap_partner_id)
        '''
        # Keep platoon, remove swap
        vehicles_to_remove_from_swap.add(veh_id)
        if swap_partner_id in conflicting_vehicles:
            vehicles_to_remove_from_swap.add(swap_partner_id)
        
    # Remove conflicts from swap_pairs
    if vehicles_to_remove_from_swap:
        swap_pairs_filtered = []
        for veh1_id, veh2_id, state1, state2 in swap_pairs:
            if veh1_id not in vehicles_to_remove_from_swap and veh2_id not in vehicles_to_remove_from_swap:
                swap_pairs_filtered.append((veh1_id, veh2_id, state1, state2))
        swap_pairs = swap_pairs_filtered
    
    # Remove conflicts from platoon_pairs
    if vehicles_to_remove_from_platoon:
        platoon_pairs_filtered = []
        for lane_veh_id, joining_veh_id, position_relation, lane_state, joining_state in platoon_pairs:
            if lane_veh_id not in vehicles_to_remove_from_platoon and joining_veh_id not in vehicles_to_remove_from_platoon:
                platoon_pairs_filtered.append((lane_veh_id, joining_veh_id, position_relation, lane_state, joining_state))
        platoon_pairs = platoon_pairs_filtered
        
    
    # -----ASSIGN SWAP RESOLUTION ROLES-----
    for veh1_id, veh2_id, state1, state2 in swap_pairs:
        veh1 = state1['veh']
        veh2 = state2['veh']
        #speed_diff = veh1['speed'] - veh2['speed']  # not used
        pos_diff = veh1['position'] - veh2['position']
        
        # Decide roles based on positions; the one ahead accelerates, the one behind decelerates
        if pos_diff >= 0:
            # veh1 is ahead and faster or equal speed
            state1['swap_role'] = 'accelerate'
            state2['swap_role'] = 'decelerate'
        else:
            # veh2 is ahead and faster or equal speed
            state2['swap_role'] = 'accelerate'
            state1['swap_role'] = 'decelerate'
        
        state1['swap_partner'] = veh2_id
        state2['swap_partner'] = veh1_id
        driving_actions.swap_partners[veh1_id] = veh2_id
        driving_actions.swap_partners[veh2_id] = veh1_id
    
    # -----ASSIGN PLATOON ROLES-----
    
    for lane_veh_id, joining_veh_id, position_relation, lane_state, joining_state in platoon_pairs:
        # Only form platoon if joining vehicle needs to change lanes and has some urgency
        if joining_state['needs_change'] and joining_state['urgency'] > 0.1:
            lane_state['platoon_role'] = 'lane_vehicle'
            lane_state['platoon_partner'] = joining_veh_id
            lane_state['platoon_relation'] = position_relation
            
            joining_state['platoon_role'] = 'joining_vehicle'
            joining_state['platoon_partner'] = lane_veh_id
            joining_state['platoon_relation'] = position_relation
            
            driving_actions.platoon_pairs[lane_veh_id] = joining_veh_id
            driving_actions.platoon_pairs[joining_veh_id] = lane_veh_id
    
    # =====SECOND PASS: GENERATE ACTIONS=====
    
    # it's interesting to ask how to iterate over vehicles
    # in reality driving decision making is continuous process and simultaneously happening for all vehicles
    # in simulation we have to iterate one by one somehow
    # naturally we want to iterate from upstream to downstream
    # we can 1) iterate over all lanes at the same time, or 2) iterate lane by lane
    # implementation 1 (not used)
    #vehicles_info_sorted = sorted(vehicles_info, key=lambda x: x['position'], reverse=True)
    #for veh in vehicles_info_sorted:

    #  implementation 2
    
    #for veh in vehicles_info:
    for lane in lanes:
        for veh in lanes[lane]:
            veh_id = veh['veh_id']
            
            # -----DEBUGGING BREAKPOINT-----
            # start from here check decision reasoning
            if traci.simulation.getTime() == 226 and veh_id in ['veh_4_213_0']:  # for debugging
                print('', end='')  # put a breakpoint here if needed
            # --------------------------------
                
            veh_type = veh['veh_type']
            position = veh['position']
            speed = veh['speed']
            current_lane = veh['current_lane']
            lane_index = int(current_lane.split('_')[-1])
            my_length = veh_length[veh_type]
            
            state = vehicle_states[veh_id]
            is_entry = state['is_entry']
            target_lane_index = state['target_lane_index']
            urgency = state['urgency']
            swap_role = state['swap_role']
            swap_partner = state['swap_partner']
            platoon_role = state['platoon_role']
            platoon_partner = state['platoon_partner']
            platoon_relation = state['platoon_relation']
            dist_to_end = get_current_lane_length(veh) - position  # distance to the end of the lane
            
            # Find leader in current lane
            current_lane_vehs = lanes[current_lane]
            vehicles_ahead = [v for v in current_lane_vehs if v['position'] > position]
            vehicles_ahead = sorted(vehicles_ahead, key=lambda x: x['position'])
            leader = vehicles_ahead[0] if vehicles_ahead else None
            leader_speed = leader['speed'] if leader else None
            if leader:
                leader_length = veh_length[leader['veh_type']]
                leader_gap = leader['position'] - position - leader_length
            else:
                leader_gap = None
            
            # Base acceleration
            accel = compute_accel(speed, veh_type, leader_speed, leader_gap, edge_info['max_speed'])
            
            # Track target lane reached
            if lane_index == target_lane_index:
                if veh_id not in driving_actions.target_lane_reached:
                    driving_actions.target_lane_reached[veh_id] = {'time': current_time, 'position': position}
            else:
                if veh_id in driving_actions.target_lane_reached:
                    del driving_actions.target_lane_reached[veh_id]
            
            # Lane change decision
            lane_changing = 0
            last_change = driving_actions.last_lane_change.get(veh_id, 0)
            time_since_change = current_time - last_change
            
            # `in_target_recently` variable checks if a vehicle has recently reached its target lane and should therefore stay there for a while
            in_target_recently = veh_id in driving_actions.target_lane_reached and \
                                position - driving_actions.target_lane_reached[veh_id]['position'] < TARGET_LANE_PATIENCE
            
            # -----HANDLE PLATOON COORDINATION (WITH DYNAMIC OFFSET)-----
            if platoon_role is not None and platoon_partner is not None and platoon_relation is not None:
                partner_state = vehicle_states.get(platoon_partner)
                if partner_state:
                    partner_veh = partner_state['veh']
                    partner_veh_type = partner_veh['veh_type']
                    pos_offset = position - partner_veh['position']
                    
                    
                    if platoon_role == 'lane_vehicle':
                        # Vehicle already in target lane
                        if platoon_relation == 'lane_ahead':
                            # Lane vehicle is ahead: ACCELERATE to pull further ahead
                            accel = max_accel[veh_type]
                        else:  # 'lane_behind'
                            # Lane vehicle is behind: DECELERATE to create gap in front
                            accel = max( - PLATOON_DECEL, -max_decel[veh_type])
                    
                    elif platoon_role == 'joining_vehicle':
                        # Vehicle that needs to join the target lane
                        direction = 1 if target_lane_index > lane_index else -1
                        target_lane_id = f"{edge_info['edge_id']}_{lane_index + direction}"
                        
                        if platoon_relation == 'lane_ahead':
                            # Calculate required offset based on vehicle lengths
                            required_offset = calculate_min_offset(partner_veh_type)
                        
                            # Lane vehicle is ahead: DECELERATE to fall behind
                            accel = max(- PLATOON_DECEL, -max_decel[veh_type])
                            
                            # Change lane when offset is sufficient (negative = we're behind)
                            if pos_offset < -required_offset and time_since_change >= LANE_CHANGE_COOLDOWN:
                                if 0 <= lane_index + direction < edge_info['lane_count']:
                                    if is_lane_change_safe(veh, target_lane_id, lanes, urgency):
                                        lane_changing = direction
                                        # Clear platoon after merge
                                        if veh_id in driving_actions.platoon_pairs:
                                            partner = driving_actions.platoon_pairs[veh_id]
                                            if partner in driving_actions.platoon_pairs:
                                                del driving_actions.platoon_pairs[partner]
                                            del driving_actions.platoon_pairs[veh_id]
                        
                        else:  # 'lane_behind'
                            # Calculate required offset based on vehicle lengths
                            required_offset = calculate_min_offset(veh_type)
                            
                            # Lane vehicle is behind: ACCELERATE to get ahead
                            accel = max_accel[veh_type]
                            
                            # Change lane when offset is sufficient (positive = we're ahead)
                            if pos_offset > required_offset and time_since_change >= LANE_CHANGE_COOLDOWN:
                                if 0 <= lane_index + direction < edge_info['lane_count']:
                                    if is_lane_change_safe(veh, target_lane_id, lanes, urgency):
                                        lane_changing = direction
                                        # Clear platoon after merge
                                        if veh_id in driving_actions.platoon_pairs:
                                            partner = driving_actions.platoon_pairs[veh_id]
                                            if partner in driving_actions.platoon_pairs:
                                                del driving_actions.platoon_pairs[partner]
                                            del driving_actions.platoon_pairs[veh_id]
            
            # -----HANDLE SWAP COORDINATION-----
            elif swap_role is not None and swap_partner is not None:
                partner_state = vehicle_states.get(swap_partner)
                if partner_state:
                    partner_veh = partner_state['veh']
                    partner_veh_type = partner_veh['veh_type']
                    partner_veh_length = veh_length[partner_veh_type]
                    partner_speed = partner_veh['speed']
                    pos_offset = position - partner_veh['position']
                    
                    direction = 1 if target_lane_index > lane_index else -1
                    target_lane_id = f"{edge_info['edge_id']}_{lane_index + direction}"
                    
                    if swap_role == 'accelerate':
                        accel = min(SWAP_COORDINATION_ACCEL, max_accel[veh_type])
                        
                        # Calculate required offset based on vehicle lengths
                        required_offset = calculate_min_offset(veh_type)
                                
                        if pos_offset > required_offset and time_since_change >= LANE_CHANGE_COOLDOWN:
                            if 0 <= lane_index + direction < edge_info['lane_count']:
                                my_safe = is_lane_change_safe(veh, target_lane_id, lanes, urgency, required_offset)
                                
                                partner_direction = 1 if partner_state['target_lane_index'] > int(partner_veh['current_lane'].split('_')[-1]) else -1
                                partner_target_lane = f"{edge_info['edge_id']}_{int(partner_veh['current_lane'].split('_')[-1]) + partner_direction}"
                                
                                if my_safe:
                                    lane_changing = direction
                    
                    elif swap_role == 'decelerate':
                        accel = max(- SWAP_COORDINATION_DECEL, -max_decel[veh_type])
                        # Calculate required offset based on vehicle lengths
                        required_offset = calculate_min_offset(partner_veh_type)
                        
                        if pos_offset < -required_offset and speed <= partner_speed and time_since_change >= LANE_CHANGE_COOLDOWN:
                            if 0 <= lane_index + direction < edge_info['lane_count']:
                                my_safe = is_lane_change_safe(veh, target_lane_id, lanes, urgency, required_offset)
                                
                                partner_direction = 1 if partner_state['target_lane_index'] > int(partner_veh['current_lane'].split('_')[-1]) else -1
                                partner_target_lane = f"{edge_info['edge_id']}_{int(partner_veh['current_lane'].split('_')[-1]) + partner_direction}"
                                
                                if my_safe:
                                    lane_changing = direction
                                            
            # -----NORMAL LANE CHANGE LOGIC-----
            elif not in_target_recently and lane_index != target_lane_index and time_since_change >= LANE_CHANGE_COOLDOWN:
                if not is_entry and urgency > 0.05:
                    direction = 1 if target_lane_index > lane_index else -1
                    target_lane_id = f"{edge_info['edge_id']}_{lane_index + direction}"
                    
                    if 0 <= lane_index + direction < edge_info['lane_count']:
                        if is_lane_change_safe(veh, target_lane_id, lanes, urgency):
                            lane_changing = direction
                            driving_actions.lane_change_intent[veh_id] = {
                                'time': current_time,
                                'direction': direction,
                                'target': target_lane_id,
                                'urgency': urgency
                            }
                        else:
                            # Gap creation logic
                            target_vehs = lanes.get(target_lane_id, [])
                            vehicles_ahead = [v for v in target_vehs if v['position'] > position]
                            vehicles_ahead = sorted(vehicles_ahead, key=lambda x: x['position'])
                            target_leader = vehicles_ahead[0] if vehicles_ahead else None
                            target_follower = next((v for v in target_vehs if v['position'] < position), None)
                            
                            if target_leader and target_follower:
                                leader_length = veh_length[target_leader['veh_type']]
                                leader_gap = target_leader['position'] - position - leader_length
                                follower_gap = position - target_follower['position'] - my_length
                                
                                if leader_gap < follower_gap:
                                    if urgency > 0.2:
                                        accel = max( - GAP_SEARCH_DECEL, -max_decel[veh_type] * 0.5)
                                else:
                                    if urgency > 0.2:
                                        accel = min( GAP_SEARCH_ACCEL, max_accel[veh_type])
                            elif target_follower:
                                if urgency > 0.2:
                                    accel = min( GAP_SEARCH_ACCEL, max_accel[veh_type])
                            elif target_leader:
                                if urgency > 0.2:
                                    accel = max( - GAP_SEARCH_DECEL, -max_decel[veh_type] * 0.5)
                            
                            # Speed matching
                            if target_vehs and urgency > 0.3:
                                avg_speed = sum(v['speed'] for v in target_vehs) / len(target_vehs)
                                speed_diff = avg_speed - speed
                                if abs(speed_diff) > 2.0:
                                    accel += speed_diff * SPEED_MATCH_FACTOR
            
            # -----COOPERATIVE BEHAVIOR-----
            # this is the cooperative behavior to help others merge in, even when no platoon or swap is formed
            for other_id, intent in driving_actions.lane_change_intent.items():
                if other_id != veh_id and other_id != swap_partner and other_id != platoon_partner:
                    other_state = vehicle_states.get(other_id)
                    if not other_state:
                        continue
                    
                    other_veh = other_state['veh']
                    other_urgency = intent.get('urgency', 0)
                    
                    if abs(other_veh['position'] - position) < COOPERATION_RANGE:
                        if intent['target'] == current_lane:
                            # other vehicle is behind us and want to merge in
                            if position > other_veh['position'] and position - other_veh['position'] < 60:  
                                if other_urgency > 0.4:
                                    # increase acceleration to create gap
                                    accel = min(accel + 1.2 * other_urgency, max_accel[veh_type])  
                            # Other vehicle is ahead of us and want to merge in
                            elif position < other_veh['position'] and other_veh['position'] - position < 60:  
                                if other_urgency > 0.4:
                                    # decrease acceleration to create gap
                                    accel = max(accel - 1.0 * other_urgency, -max_decel[veh_type] * 0.4)  
                        
                        other_lane_idx = int(other_veh['current_lane'].split('_')[-1])
                        if abs(other_lane_idx - lane_index) == 1:
                            if abs(position - other_veh['position']) < 30 and other_urgency > 0.5:
                                if position > other_veh['position']:
                                    accel = min(accel + 0.8, max_accel[veh_type] * 0.7)
                                else:
                                    accel = max(accel - 0.6, -max_decel[veh_type] * 0.3)
        
                
            # -----FINAL ACCELERATION CONSTRAINTS-----
            accel = max(-max_decel[veh_type], min(max_accel[veh_type], accel))
            if accel > 0:
                accel = max(MIN_ACCEL, accel)
            
            # -----UPDATE MEMORY-----
            if lane_changing != 0:
                driving_actions.last_lane_change[veh_id] = current_time
                
                # Clear swap partner after lane change
                if veh_id in driving_actions.swap_partners:
                    partner = driving_actions.swap_partners[veh_id]
                    if partner in driving_actions.swap_partners:
                        del driving_actions.swap_partners[partner]
                    del driving_actions.swap_partners[veh_id]
            
            # -----APPEND FINAL ACTION-----
            actions.append({
                'veh_id': veh_id,
                'acceleration': accel,
                'lane_changing': lane_changing
            })
    

    # =====CAR-FOLLOWING SAFETY CHECK=====
    # Car-following may not be safe; say, if the leader brakes hard or just not accelerating fast enough
    # (Note: compute_accel function does not consider future leader deceleration)
    # For each lane, start from the furthest vehicle (largest position), check if the following vehicle can maintain safe headway at next time step
    # for headway, we use MIN_GAP and SAFE_TTC
    
    # Process each lane separately
    for lane_id, lane_vehicles in lanes.items():
        if not lane_vehicles:
            continue
        
        # Vehicles are already sorted by position (descending - front to back)
        # Start from the front and check each follower
        for i in range(len(lane_vehicles) - 1):
            leader_veh = lane_vehicles[i]
            follower_veh = lane_vehicles[i + 1]
            
            leader_id = leader_veh['veh_id']
            follower_id = follower_veh['veh_id']
            
            # Get actions for both vehicles
            leader_action = next((a for a in actions if a['veh_id'] == leader_id), None)
            follower_action = next((a for a in actions if a['veh_id'] == follower_id), None)
            
            if not leader_action and not follower_action:
                continue
            
            '''it's interesting to see that no skipping makes it safer
            # Skip if either vehicle is changing lanes (they won't be in the same lane next step)
            if leader_action['lane_changing'] != 0 or follower_action['lane_changing'] != 0:
                continue
            '''
            
            # Get vehicle properties
            leader_type = leader_veh['veh_type']
            follower_type = follower_veh['veh_type']
            leader_length = veh_length[leader_type]
            follower_length = veh_length[follower_type]
            
            # Current state
            leader_pos = leader_veh['position']
            follower_pos = follower_veh['position']
            leader_speed = leader_veh['speed']
            follower_speed = follower_veh['speed']
            
            # Current gap (without leader length)
            current_gap = leader_pos - follower_pos - leader_length
            
            # Get accelerations from actions
            leader_accel = leader_action['acceleration']
            follower_accel = follower_action['acceleration']
            
            # time step is 1 second
            # Predict next positions and speeds (kinematic equations)
            leader_pos_next = leader_pos + leader_speed + 0.5 * leader_accel
            follower_pos_next = follower_pos + follower_speed + 0.5 * follower_accel
            
            # Gap at next time step
            gap_next = leader_pos_next - follower_pos_next - leader_length
            
            if gap_next < MIN_GAP:
                gap_to_add = MIN_GAP - gap_next
                accel_adjustment = gap_to_add * 2
                # Calculate new acceleration (ensure it doesn't exceed max deceleration)
                new_accel = follower_accel - accel_adjustment
                if new_accel >= -max_decel[follower_type]:
                    new_accel = max(new_accel, -max_decel[follower_type])
                    # Update follower's acceleration
                    follower_action['acceleration'] = new_accel
                else:
                    follower_action['acceleration'] = -max_decel[follower_type]
                    leader_accel_needed = (- new_accel) - max_decel[follower_type] - 0.5  # in this case, we allow smaller buffer; namely MIN_GAP decrease by 1 meter
                    leader_action['acceleration'] = leader_accel + leader_accel_needed
                
    
    # =====MERGING SAFETY CHECK=====
    # Check before returning actions
    # When a vehicle is driving straight on a lane, it may not be aware of future merging intentions
    # In that case, it may collide with the merging vehicle at next time step
    # To prevent this, we check if there is any vehicle in adjacent lanes that may merge into our lane soon
    # If so, we adjust our acceleration to maintain a safe distance, namely we're being polite drivers
    
    # Build a map of lane changes from actions
    lane_change_map = {}  # {veh_id: (direction, current_lane, target_lane)}
    for action in actions:
        if action['lane_changing'] != 0:
            veh_id = action['veh_id']
            veh = next(v for v in vehicles_info if v['veh_id'] == veh_id)
            current_lane = veh['current_lane']
            lane_idx = int(current_lane.split('_')[-1])
            target_lane_idx = lane_idx + action['lane_changing']
            target_lane = f"{edge_info['edge_id']}_{target_lane_idx}"
            lane_change_map[veh_id] = (action['lane_changing'], current_lane, target_lane)
    
    # Check each vehicle not changing lanes for potential collision with merging vehicles
    for i, action in enumerate(actions):
        if action['lane_changing'] == 0:  # Vehicle is staying in current lane
            veh_id = action['veh_id']
            veh = next(v for v in vehicles_info if v['veh_id'] == veh_id)
            veh_type = veh['veh_type']
            position = veh['position']
            speed = veh['speed']
            current_lane = veh['current_lane']
            my_length = veh_length[veh_type]
            
            # Check adjacent lanes for vehicles merging into our lane
            lane_idx = int(current_lane.split('_')[-1])
            
            for other_id, (direction, other_current_lane, other_target_lane) in lane_change_map.items():
                if other_target_lane == current_lane:  # Other vehicle is merging into our lane
                    other_veh = next(v for v in vehicles_info if v['veh_id'] == other_id)
                    other_position = other_veh['position']
                    other_speed = other_veh['speed']
                    other_type = other_veh['veh_type']
                    other_length = veh_length[other_type]
                    
                    # Calculate potential conflict zone
                    position_diff = abs(position - other_position)
                    
                    # Check if vehicles are close enough to be a concern
                    if position_diff < 30.0:  # Within 30m range
                        # Determine relative positions after merge
                        if other_position > position:
                            # Other vehicle will be ahead after merging
                            gap = other_position - position - other_length
                            rel_speed = speed - other_speed
                            
                            # Check if we're approaching them
                            if rel_speed > 0 and gap < 20.0:
                                # Calculate required deceleration to maintain safe distance
                                ttc = gap / rel_speed if rel_speed > 0 else float('inf')
                                if ttc < SAFE_TTC:
                                    # Reduce our acceleration to be polite
                                    politeness_decel = min(1.5, rel_speed / SAFE_TTC)
                                    action['acceleration'] = max(
                                        action['acceleration'] - politeness_decel,
                                        -max_decel[veh_type] * 0.5
                                    )
                        
                        else:
                            # Other vehicle will be behind after merging
                            gap = position - other_position - my_length
                            rel_speed = other_speed - speed
                            
                            # Check if they're approaching us from behind
                            if rel_speed > 0 and gap < 20.0:
                                # Speed up slightly to create space
                                ttc = gap / rel_speed if rel_speed > 0 else float('inf')
                                if ttc < SAFE_TTC:
                                    politeness_accel = min(1.0, rel_speed / SAFE_TTC * 0.5)
                                    action['acceleration'] = min(
                                        action['acceleration'] + politeness_accel,
                                        max_accel[veh_type] * 0.8
                                    )

    # =====SIMULTANEOUS MERGING SAFETY CHECK=====
    # Check before returning actions
    # Check if multiple vehicles are trying to merge into the same position
    # If collision is imminent, cancel the merge for the vehicle with lower urgency
    
    # Group vehicles by their target lanes
    merging_by_target = {}  # {target_lane: [(veh_id, position, urgency, action_index)]}
    
    for i, action in enumerate(actions):
        if action['lane_changing'] != 0:
            veh_id = action['veh_id']
            veh = next(v for v in vehicles_info if v['veh_id'] == veh_id)
            position = veh['position']
            urgency = vehicle_states[veh_id]['urgency']
            
            current_lane = veh['current_lane']
            lane_idx = int(current_lane.split('_')[-1])
            target_lane_idx = lane_idx + action['lane_changing']
            target_lane = f"{edge_info['edge_id']}_{target_lane_idx}"
            
            if target_lane not in merging_by_target:
                merging_by_target[target_lane] = []
            
            merging_by_target[target_lane].append((veh_id, position, urgency, i))
    
    # Check each target lane for simultaneous merges
    for target_lane, merging_vehicles in merging_by_target.items():
        if len(merging_vehicles) > 1:
            # Sort by position to check for potential collisions
            merging_vehicles.sort(key=lambda x: x[1])  # Sort by position
            
            # Check each pair of adjacent merging vehicles
            for j in range(len(merging_vehicles) - 1):
                veh1_id, pos1, urgency1, idx1 = merging_vehicles[j]
                veh2_id, pos2, urgency2, idx2 = merging_vehicles[j + 1]
                
                veh1 = next(v for v in vehicles_info if v['veh_id'] == veh1_id)
                veh2 = next(v for v in vehicles_info if v['veh_id'] == veh2_id)
                
                veh1_length = veh_length[veh1['veh_type']]
                veh2_length = veh_length[veh2['veh_type']]
                
                # Calculate gap after both vehicles merge
                gap = pos2 - pos1 - veh2_length
                
                # Calculate relative speed
                rel_speed = veh1['speed'] - veh2['speed']
                
                # Check if collision is imminent
                collision_imminent = False
                
                if gap < OFFSET_BUFFER * 2:  # Gap is too small
                    collision_imminent = True
                elif rel_speed > 0 and gap > 0:  # veh1 is approaching veh2
                    ttc = gap / rel_speed
                    if ttc < SAFE_TTC * 0.5:  # Very short TTC
                        collision_imminent = True
                
                if collision_imminent:
                    # Cancel the merge for the vehicle with lower urgency
                    if urgency1 < urgency2:
                        # Cancel veh1's lane change
                        actions[idx1]['lane_changing'] = 0
                        # Apply gentle deceleration to create more space
                        veh1_type = veh1['veh_type']
                        actions[idx1]['acceleration'] = max(
                            actions[idx1]['acceleration'] - 1.0,
                            -max_decel[veh1_type] * 0.4
                        )
                    elif urgency2 < urgency1:
                        # Cancel veh2's lane change
                        actions[idx2]['lane_changing'] = 0
                        # Apply gentle deceleration to create more space
                        veh2_type = veh2['veh_type']
                        actions[idx2]['acceleration'] = max(
                            actions[idx2]['acceleration'] - 1.0,
                            -max_decel[veh2_type] * 0.4
                        )
                    else:
                        # Equal urgency - cancel the one behind (veh1)
                        actions[idx1]['lane_changing'] = 0
                        veh1_type = veh1['veh_type']
                        actions[idx1]['acceleration'] = max(
                            actions[idx1]['acceleration'] - 1.0,
                            -max_decel[veh1_type] * 0.4
                        )
        
    # =====RETURN ACTIONS=====
    return actions             